# ARC-AGI-2 Solver — Qwen3.8 27B FP8 on 4×L4 (vLLM)

A complete evaluation pipeline for **ARC-AGI-2**: serve Qwen3.8 27B (official FP8
checkpoint) locally with vLLM across 4×L4 GPUs, solve every evaluation task in
parallel under a hard time budget, then aggregate and score. Everything runs inside
this notebook — no API keys, no internet.

---


# ARC-AGI-2 求解器 — Qwen3.8 27B FP8 @ 4×L4 (vLLM)

针对 **ARC-AGI-2** 的完整评测流水线：用 vLLM 在 4×L4 GPU 上本地部署 Qwen3.8 27B
（官方 FP8 checkpoint），在硬性时间预算下并行求解全部评测任务，然后汇总打分。
整个流程都跑在本 notebook 内 —— 不需要 API key，也不需要联网。

---


## How to use


## 使用方法


### 1. Set up the Kaggle notebook

Set the accelerator to **GPU 4×L4** and leave internet **off**. Then attach three inputs:

| Input | Mounted at | Used for |
|-------|-----------|----------|
| ARC Prize 2026 — ARC-AGI-2 | `/kaggle/input/competitions/arc-prize-2026-arc-agi-2/` | tasks + solutions |
| Qwen3.8 27B FP8 (official) | `/kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1` | model weights |
| `arc3-vllm-h100-wheelhouse-v3` | `/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3` | prebuilt wheels + `requirements.lock` |

The wheelhouse exists because Kaggle runs without internet. If you *do* have internet,
you can replace section 1 with a plain `pip install vllm`.


### 1. 配置 Kaggle notebook

加速器选 **GPU 4×L4**，外网保持**关闭**。然后挂载三个 input：

| Input | 挂载路径 | 用途 |
|-------|---------|------|
| ARC Prize 2026 — ARC-AGI-2 | `/kaggle/input/competitions/arc-prize-2026-arc-agi-2/` | 题目与标准答案 |
| Qwen3.8 27B FP8（官方） | `/kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1` | 模型权重 |
| `arc3-vllm-h100-wheelhouse-v3` | `/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3` | 预编译 wheel + `requirements.lock` |

需要 wheelhouse 是因为 Kaggle 运行时没有外网。如果你**确实**能联网，
可以把第 1 节换成一普通的 `pip install vllm`。


### 2. Run the cells in order

`Run All`. Section 1 may tell you to restart the kernel once — that happens when the
wheelhouse `torch` does not match the L4 driver, and it is expected. If it does:
`Kernel → Restart`, then `Run All` again.

Section 2 takes **5–12 minutes** to load the weights and capture CUDA graphs. It then
verifies the attention backend that actually took effect and fires one smoke request,
so a backend misconfiguration shows up immediately rather than an hour into the run.


### 2. 按顺序运行

`Run All`。第 1 节可能会提示你重启内核一次 —— 当 wheelhouse 的 `torch` 与 L4 驱动不匹配
时就会这样，属于预期行为。如果出现，执行 `Kernel → Restart`，再 `Run All` 一遍。

第 2 节需要 **5–12 分钟**加载权重并捕获 CUDA graph。之后它会验证实际生效的 attention
后端，并发一条冒烟请求 —— 这样后端配错会立刻暴露，而不是跑了一小时才发现。


### 3. Choose a mode and concurrency

```python
run(mode='eval', n=32)     # solve + aggregate + score
run(mode='submit', n=32)   # solve + aggregate only
```

- `eval` scores the run against the solutions file. `submit` never reads it — use it
  when you only want a `submission.json` to upload.
- `n` is how many tasks are solved concurrently. Keep it at or below `--max-num-seqs`
  in section 2 (32 here).
- An unsolved task costs nothing but a fallback grid, so running wide is cheap; running
  long is what the deadline protects against.


### 3. 选择模式与并发数

```python
run(mode='eval', n=32)     # 求解 + 汇总 + 打分
run(mode='submit', n=32)   # 只求解 + 汇总
```

- `eval` 会拿标准答案打分。`submit` 完全不读答案文件 —— 当你只想要一份用来上传的
  `submission.json` 时用它。
- `n` 是同时求解的任务数。不要超过第 2 节的 `--max-num-seqs`（这里是 32）。
- 一个没解出来的任务只损失一个兜底网格，所以开宽并发很便宜；
  真正需要防的是跑太久，那由时限兜住。


### 4. The 11.5-hour deadline

The pipeline is built to be cut off safely, because a Kaggle session has a hard limit
and hitting it mid-run is the normal case, not the exception.

- The clock starts at the **first cell of the notebook** and defaults to **690 minutes
  (11h30m)**.
- Once it passes, no new task is started; requests already in flight are cut off by
  their own timeout, which is itself clamped to the remaining budget.
- Unfinished tasks are then filled in by the **rule-based fallback** (below), so the
  submission is still complete and submittable.

To change it, set `RUN_START_TS` / `DEADLINE_MIN` in a cell *before* section 4.

Independently of the deadline, `submission.json` is never left incomplete: it is written
in full *before* inference starts, updated in place as answers arrive, and repaired slot
by slot at the end. See section 4.


### 4. 11.5 小时时限

整条流水线是按"可以被安全掐断"来设计的，因为 Kaggle 会话有硬上限，
跑到一半被切断是常态而不是意外。

- 计时从 **notebook 的第一个 cell** 开始，默认 **690 分钟（11 小时 30 分）**。
- 一旦到点，就不再开新任务；在飞的请求被各自的超时掐断，
  而那个超时本身也被压到剩余预算之内。
- 没跑完的任务随后由**基于规则的兜底**补齐（见下），
  所以提交文件依然是完整、可提交的。

要改的话，在第 4 节**之前**的 cell 里设置 `RUN_START_TS` / `DEADLINE_MIN`。

与时限无关地，`submission.json` 绝不会残缺：它在推理开始**之前**就完整写出，
随答案产出就地更新，结束时逐槽位修复兜底。详见第 4 节。


### 5. Outputs

Everything lands in `/kaggle/working/`:

| Path | Contents |
|------|----------|
| `submission.json` | The aggregated submission. Always complete — every slot holds a valid grid, never `null`. **This is what you upload.** |
| `score.json` | Eval-mode report: total score plus a per-task breakdown. |
| `output/result_<taskid>.json` | One file per solved task. Its existence is what `SKIP_EXISTING` checks. |
| `output/solve.log` | Timestamped per-task log: prompt size, each retry, failure reason. |
| `output/raw_<taskid>.txt` | Raw model output — written **only** when parsing failed. Start here when debugging. |
| `thinking.log` | Every prompt and its reasoning trace. Large; useful when iterating on prompts. |
| `vllm_server.log` | Server stdout/stderr. |

A failed task writes **no** result file. That is deliberate: it is what makes
`SKIP_EXISTING` a correct resume mechanism, and what tells aggregation that the task
needs a fallback.


### 5. 输出文件

一切都落在 `/kaggle/working/`：

| 路径 | 内容 |
|------|------|
| `submission.json` | 汇总后的提交文件。永远完整 —— 每个槽位都是合法网格，绝不为 `null`。**要上传的就是它。** |
| `score.json` | 评估模式报告：总分，以及逐任务明细。 |
| `output/result_<taskid>.json` | 每个已求解任务一个文件。文件是否存在就是 `SKIP_EXISTING` 的判断依据。 |
| `output/solve.log` | 带时间戳的逐任务日志：prompt 大小、每次重试、失败原因。 |
| `output/raw_<taskid>.txt` | 模型原始输出 —— **仅**在解析失败时写。调试从这里开始。 |
| `thinking.log` | 每次的 prompt 和对应推理过程。体积大；迭代 prompt 时有用。 |
| `vllm_server.log` | 服务的 stdout/stderr。 |

失败的任务**不写**结果文件。这是刻意的：正因如此 `SKIP_EXISTING` 才是正确的断点续跑
机制，也正因如此汇总阶段才知道该任务需要兜底。


### 6. Tuning

All knobs are constants at the top of section 4:

| Constant | Default | Effect |
|----------|---------|--------|
| `N_PARALLEL` | `16` | Concurrent tasks. Must be ≤ `--max-num-seqs`. |
| `MODE` | `"eval"` | Default when `run()` is called with no arguments. |
| `DEADLINE_MIN` | `690` | Minutes from notebook start until the hard stop. |
| `SKIP_EXISTING` | `True` | Resume: skip tasks that already have a result file. |
| `MAX_RETRIES` | `0` | Attempts per task, beyond the first. `0` means one attempt. |
| `PROGRESS_REFRESH` | `5.0` | Progress line interval; `0` disables the monitor thread. |

`MAX_RETRIES` is 0 on purpose: with `temperature=0.0`, a retry reproduces the same
answer, so a second attempt rarely helps and always costs time you may not have.

Switching datasets is also just a constant: point `CHALLENGES_FILE` at the training or
test split and use `mode='submit'` where no solutions exist.


### 6. 调参

所有旋钮都是第 4 节顶部的常量：

| 常量 | 默认值 | 作用 |
|------|--------|------|
| `N_PARALLEL` | `16` | 并发任务数。必须 ≤ `--max-num-seqs`。 |
| `MODE` | `"eval"` | 不带参数调用 `run()` 时的默认模式。 |
| `DEADLINE_MIN` | `690` | 从 notebook 启动算起，多少分钟后硬停。 |
| `SKIP_EXISTING` | `True` | 断点续跑：已有结果文件的任务跳过。 |
| `MAX_RETRIES` | `0` | 首次之外的重试次数。`0` 表示只尝试一次。 |
| `PROGRESS_REFRESH` | `5.0` | 进度打印间隔；`0` 关闭监控线程。 |

`MAX_RETRIES` 设成 0 是刻意的：`temperature=0.0` 下重试会复现同一个答案，
所以第二次尝试很少有用，却一定要花你可能没有的时间。

换数据集也只是改常量：把 `CHALLENGES_FILE` 指向训练集或测试集，
在没有标准答案的划分上用 `mode='submit'`。


### 7. Resuming and partial reruns

Model calls are the expensive part, so they are kept separate from everything else.

- **Interrupted run** → run the same cell again. Solved tasks are skipped, the rest retry.
- **Retry the failures** → delete the specific `output/result_<taskid>.json` files.
  Tasks that never succeeded have no file, so they retry on their own.
- **Changed scoring or aggregation?** → no model calls needed:
  ```python
  aggregate_to_submission()   # rebuild submission.json from existing results
  score_existing()            # rescore
  ```
- **Changed the prompt?** → cached results are stale. Delete `output/` before re-running.


### 7. 断点续跑与局部重跑

模型调用是唯一昂贵的一环，所以设计上把它和其余部分解耦了。

- **中途中断** → 重跑同一个 cell。已求解的任务跳过，其余重试。
- **只重试失败的任务** → 删掉对应的 `output/result_<taskid>.json`。
  从未成功的任务本来就没有文件，会自动重试。
- **改了打分或汇总逻辑？** → 不需要任何模型调用：
  ```python
  aggregate_to_submission()   # 用已有结果重建 submission.json
  score_existing()            # 重新打分
  ```
- **改了 prompt？** → 缓存的结果已过期。重跑前删掉 `output/`。


### 8. Running outside the notebook

Section 4 is a self-contained module. Copy that one cell into `arc_pipeline.py` and it
runs standalone — that is what the `argparse` block and the `__main__` guard are for:

```bash
python arc_pipeline.py --mode eval   --n 32   # solve + aggregate + score
python arc_pipeline.py --mode submit --n 32   # solve + aggregate only
python arc_pipeline.py --only-aggregate       # rebuild submission.json, no model calls
python arc_pipeline.py --only-score           # score the existing submission.json
```

The `__main__` guard checks for an IPython kernel before running `argparse`, because in
a notebook `sys.argv` holds the kernel's own arguments, not yours.

To use a different model, replace `chat()` in section 4 — or override it without
editing the file:

```python
import arc_pipeline as A
A.chat = my_chat_function
A.run(mode="eval", n=32)
```


### 8. 在 notebook 之外使用

第 4 节是一个自包含模块。把那一个 cell 拷成 `arc_pipeline.py` 就能独立运行 ——
`argparse` 块和 `__main__` 守卫就是为此准备的：

```bash
python arc_pipeline.py --mode eval   --n 32   # 求解 + 汇总 + 打分
python arc_pipeline.py --mode submit --n 32   # 只求解 + 汇总
python arc_pipeline.py --only-aggregate       # 重建 submission.json，无模型调用
python arc_pipeline.py --only-score           # 给已有 submission.json 打分
```

`__main__` 守卫会在运行 `argparse` 前先检测 IPython 内核，因为在 notebook 里
`sys.argv` 装的是内核自己的参数，不是你的。

想换模型，就替换第 4 节的 `chat()` —— 或者不改文件，直接覆盖：

```python
import arc_pipeline as A
A.chat = my_chat_function
A.run(mode="eval", n=32)
```


### 9. Troubleshooting

| Symptom | Cause | Fix |
|---------|-------|-----|
| `GPU is visible but torch has no CUDA` | wheelhouse `torch` does not match the L4 driver | The cell reinstalls cu126 automatically — **restart the kernel**, then Run All |
| Crash with `illegal memory access` | FlashInfer FP8 kernels need SM90+; the L4 is SM89 | Keep `ATTN_BACKEND = "TRITON_ATTN"` **and** `VLLM_FORCE_ATTN_BACKEND` |
| Log still shows `FLASHINFER` | the MTP drafter did not inherit the CLI flag | Check `VLLM_FORCE_ATTN_BACKEND` is set; it is the only thing that pins the drafter |
| Output is garbled or repetitive | MTP acceptance rate is 0% — a separate bug | Lower `NUM_SPEC_TOKENS` to 2, or drop `--speculative-config` entirely |
| Startup OOM | `--max-model-len` or `--gpu-memory-utilization` too high | Lower `--max-model-len` first |
| `connection refused` in `output/solve.log` | the vLLM process died mid-run | Check `vllm_server.log`, then re-run — finished tasks are skipped |
| Many tasks abandoned as `deadline` | the run needed more than `DEADLINE_MIN` | Raise `DEADLINE_MIN`, or raise `n` to get more done in the same window |
| Score is 0 | `submission.json` is empty or malformed | Check the `null` count printed by aggregation — it should be 0 |

When a task fails, `output/solve.log` and `output/raw_<taskid>.txt` between them tell
you which of the three failure classes it was: transport (the server), extraction (no
JSON produced), or validation (JSON present but the grid was wrong).

---


### 9. 故障排查

| 现象 | 原因 | 处理 |
|------|------|------|
| `GPU is visible but torch has no CUDA` | wheelhouse 的 `torch` 与 L4 驱动不匹配 | cell 会自动重装 cu126 —— **重启内核**再 Run All |
| 崩溃并报 `illegal memory access` | FlashInfer 的 FP8 kernel 需要 SM90+，L4 是 SM89 | 保留 `ATTN_BACKEND = "TRITON_ATTN"` **并且**保留 `VLLM_FORCE_ATTN_BACKEND` |
| 日志里仍有 `FLASHINFER` | MTP 的 drafter 没有继承 CLI 参数 | 检查 `VLLM_FORCE_ATTN_BACKEND` 是否设置 —— 它是唯一能钉住 drafter 的东西 |
| 输出乱码或重复 | MTP 接受率为 0%，是另一个独立 bug | 把 `NUM_SPEC_TOKENS` 降到 2，或干脆去掉 `--speculative-config` |
| 启动 OOM | `--max-model-len` 或 `--gpu-memory-utilization` 太高 | 先降 `--max-model-len` |
| `output/solve.log` 里出现 `connection refused` | vLLM 进程中途死了 | 查 `vllm_server.log`，然后重跑 —— 已完成的任务会跳过 |
| 大量任务被标记为 `deadline` | 需要的时间超过了 `DEADLINE_MIN` | 调大 `DEADLINE_MIN`，或调大 `n` 在同样窗口里做更多 |
| 分数为 0 | `submission.json` 为空或格式非法 | 看汇总时打印的 `null` 计数 —— 它应该是 0 |

当一个任务失败时，`output/solve.log` 和 `output/raw_<taskid>.txt` 合起来能告诉你
它属于三类失败中的哪一类：传输层（服务的问题）、提取层（没产出 JSON）、
还是校验层（有 JSON 但网格不对）。

---


## What each section does

| # | Section | What it does |
|---|---------|--------------|
| 1 | Environment setup | Installs vLLM and dependencies from the wheelhouse, validates CUDA, works around the flash-attn / L4 incompatibility |
| 2 | Launch the vLLM server | Starts the server with the FP8 KV cache and MTP, pins the attention backend, then verifies it |
| 3 | Wait for readiness | Polls until the engine is genuinely serving, then prints the endpoint |
| 4 | The pipeline | The substance: model client, prompt builder, deadline, parallel solver, fallback, scorer |
| 5 | Run it | The one-line entry point |

---


## 各部分做什么

| # | 小节 | 作用 |
|---|------|------|
| 1 | 环境安装 | 从 wheelhouse 安装 vLLM 及依赖，校验 CUDA，绕过 flash-attn 与 L4 的不兼容 |
| 2 | 启动 vLLM 服务 | 带 FP8 KV cache 与 MTP 启动服务，钉住 attention 后端，然后验证它 |
| 3 | 等待就绪 | 轮询直到引擎真正开始服务，再打印端点 |
| 4 | 流水线 | 主体：模型客户端、prompt 构造、时限、并行求解、兜底、打分 |
| 5 | 运行 | 一行入口 |

---


## What "solving a task" means

ARC-AGI tasks are few-shot grid transformations. Each task ships a handful of `train`
input/output grid pairs, then one or more `test` inputs whose outputs you must predict.
Grids are 2-D arrays of color indices `0`–`9`.

This pipeline serializes each task into a compact text form, asks the model for a
strict JSON answer, and validates and repairs whatever comes back. The attempt-key
rules, the deadline, and the fallback are all covered in section 4.

## 这里的"求解一个任务"是什么意思

ARC-AGI 任务是少样本的网格变换。每个任务给出若干组 `train` 输入/输出网格对，
然后有若干个 `test` 输入，需要你预测它们的输出。网格是颜色索引 `0`–`9` 组成的二维数组。

本流水线把每个任务序列化成一段紧凑的文本，要求模型返回严格的 JSON，
并对返回内容做校验和修复。attempt 键的规则、时限机制和兜底逻辑都在第 4 节。


---


# 1. Environment setup

Three things happen here, in order.

**Install from the wheelhouse.** `pip install --no-index --find-links <dir>` installs
the exact pinned dependency set from `requirements.lock`, with no network access.
Kaggle notebooks have internet disabled, so this is the only way to get vLLM in.

**Validate CUDA.** The wheelhouse was built for H100, and its `torch` sometimes links
against a CUDA version the L4 driver cannot load. This block detects that case —
`nvidia-smi` sees a GPU but `torch.cuda.is_available()` is `False` — and reinstalls
the matching cu126 build. If it triggers, **the kernel must be restarted**: an
already-imported `torch` cannot be swapped out in place. The cell exits with a message
telling you so.

**Handle flash-attn.** L4/T4/K80 have no prebuilt flash-attn wheels and building from
source always fails there, so the attempt is skipped — saving roughly five minutes of
guaranteed-futile compilation. When flash-attn genuinely is unavailable, vLLM's
attention layer needs a one-line patch, because it forwards a `layer_idx` kwarg that
the fallback implementation rejects. The patch is applied directly to `attention.py`,
and a `sitecustomize.py` shim is dropped on `PYTHONPATH` as a second line of defence.

> This cell is idempotent — the patch checks for its own marker before writing, so
> re-running it is safe.

# 1. 环境安装

这里按顺序做三件事。

**从 wheelhouse 安装。** `pip install --no-index --find-links <dir>` 按
`requirements.lock` 装上完全一致的锁定依赖集，全程不访问网络。Kaggle notebook 禁用了
外网，所以这是在 Kaggle 里装上 vLLM 的唯一办法。

**校验 CUDA。** wheelhouse 是为 H100 构建的，它带的 `torch` 有时链接的 CUDA 版本
L4 驱动加载不了。这段代码会检测这种情况 —— `nvidia-smi` 能看到 GPU，
但 `torch.cuda.is_available()` 是 `False` —— 并重装匹配的 cu126 版本。
一旦触发，**必须重启内核**：已经 import 的 `torch` 无法原地替换。
cell 会直接退出并提示你这么做。

**处理 flash-attn。** L4/T4/K80 没有预编译的 flash-attn wheel，从源码编译必定失败，
所以在这类卡上直接跳过 —— 省下约 5 分钟注定白费的编译。当 flash-attn 确实不可用时，
vLLM 的 attention 层需要打一行补丁，因为它会往回退实现传一个不被接受的 `layer_idx`
参数。补丁直接打到 `attention.py` 上，另外在 `PYTHONPATH` 上放一个 `sitecustomize.py`
作为双保险。

> 这个 cell 是幂等的 —— 补丁在写入前会先检查自己的标记，重复运行是安全的。


In [ ]:
import os, sys, subprocess, glob

# Install vLLM and its pinned dependency set from the local wheelhouse. Kaggle runs
# without internet, so --no-index + --find-links is the only way to get them in.
# 从本地 wheelhouse 安装 vLLM 及其锁定版本的依赖集。Kaggle 没有外网，
# 所以 --no-index + --find-links 是唯一的安装途径。
wheel_dir = "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"
req_file = os.path.join(wheel_dir, "requirements.lock")

print("=== Step 1: Install from wheelhouse ===")
cmd = [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-index",
       "--find-links", wheel_dir, "--requirement", req_file]
proc = subprocess.run(cmd, text=True, capture_output=True)
if proc.stdout: print(proc.stdout[-2000:])
if proc.returncode != 0:
    if proc.stderr: print(proc.stderr[-2000:])
    raise SystemExit(f"pip install failed with code {proc.returncode}")

# The wheelhouse was built for H100, and its torch may not match the L4 driver. When
# that happens the GPU is visible to nvidia-smi but invisible to torch; the fix is a
# cu126 reinstall, after which the kernel MUST be restarted, because an already
# imported torch cannot be swapped out in place.
# wheelhouse 是为 H100 构建的，它带的 torch 可能与 L4 驱动不匹配。
# 此时 nvidia-smi 能看到 GPU，torch 却看不到；解决办法是重装 cu126 版本，
# 之后必须重启内核，因为已经 import 的 torch 无法原地替换。
#
# This import must come after the install above, or it picks up the pre-install torch.
# 这个 import 必须在上面安装完成之后执行，否则拿到的是安装前的旧版 torch。
import torch
print(f"torch: {torch.__version__} | CUDA available: {torch.cuda.is_available()} | CUDA ver: {torch.version.cuda}")

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
gpu_visible = smi.returncode == 0 and "NVIDIA" in smi.stdout

if not torch.cuda.is_available():
    if gpu_visible:
        print("⚠️ GPU is visible but torch has no CUDA — the wheelhouse torch is incompatible with the L4 driver!")
        print("   Reinstalling the cu126 build of torch (L4-compatible)...")
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-cache-dir",
             "torch==2.10.0", "torchvision==0.25.0", "torchaudio==2.10.0",
             "--index-url", "https://download.pytorch.org/whl/cu126"],
            capture_output=True, text=True)
        print((r.stdout or r.stderr)[-1500:])
        print("✅ torch reinstalled as the cu126 build.")
        print("⚠️ The running kernel still holds the old torch — do Kernel → Restart, then Run All from the top.")
        print("⚠️ If vLLM later fails with a symbol error, vLLM 0.19 needs cu128: change the index-url above to cu128.")
        raise SystemExit("torch reinstalled — restart the kernel and re-run")
    else:
        # No GPU at all: the accelerator is not enabled for this session.
        # 完全没有 GPU：本次会话没有启用加速器。
        print("❌ No GPU detected (nvidia-smi produced no output)!")
        print("   Confirm the Kaggle accelerator is set to a GPU (e.g. 4×L4), then restart the session.")
        print(smi.stdout[:1000])
        raise SystemExit("no GPU available")
else:
    print(f"✅ CUDA OK: {torch.cuda.get_device_name(0)}")
    print("   nvidia-smi:")
    for ln in smi.stdout.splitlines()[:8]:
        print("   ", ln)

import vllm
print(f"vLLM version: {vllm.__version__}")

# L4/T4/K80/P4 have no prebuilt flash-attn wheels and a source build always fails on
# them, so skip the attempt entirely — it saves about 5 minutes of futile compiling.
# L4/T4/K80/P4 没有预编译的 flash-attn wheel，从源码编译必定失败，所以直接跳过，
# 省下约 5 分钟注定白费的编译时间。
print("\n=== Step 2: flash-attn detection ===")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
flash_attn_ready = False

try:
    import flash_attn
    flash_attn_ready = True
    print(f"flash-attn {flash_attn.__version__} is already available")
except Exception:
    pass

if not flash_attn_ready:
    if any(x in gpu_name for x in ["L4", "T4", "K80", "P4"]):
        print(f"⚠️ GPU {gpu_name} is not flash-attn compatible — skipping the build (~5 minutes saved)")
    else:
        # Other GPUs may still build it, so try briefly and fall through on failure.
        # 其他型号的卡有可能编译成功，简单试一下，失败就继续往下走。
        print(f"Attempting a flash-attn build (GPU: {gpu_name})...")
        fix_cmd = [sys.executable, "-m", "pip", "install", "--no-build-isolation",
                   "flash-attn>=2.7.0", "--no-deps"]
        try:
            result = subprocess.run(fix_cmd, text=True, capture_output=True, timeout=60)
            if result.returncode == 0:
                flash_attn_ready = True
                print("flash-attn built successfully")
            else:
                print("flash-attn build failed — falling back to the code-level patch")
        except subprocess.TimeoutExpired:
            print("flash-attn build timed out — falling back to the code-level patch")

# With flash-attn unavailable, vLLM's attention layer passes a layer_idx kwarg that
# the fallback implementation does not accept. Two fixes are applied: patch the call
# site in attention.py, and drop a sitecustomize.py shim on PYTHONPATH as a backstop.
# 没有 flash-attn 时，vLLM 的 attention 层会传一个回退实现不接受的 layer_idx 参数。
# 这里做两件事：给 attention.py 的调用点打补丁，并在 PYTHONPATH 上放一个
# sitecustomize.py 作为兜底。
#
# The patch is idempotent — the marker check makes re-running this cell safe.
# 这个补丁是幂等的 —— 标记检查让重复运行这个 cell 是安全的。
if not flash_attn_ready:
    print("\n=== Step 3: Patch attention.py ===")
    attn_candidates = (
        glob.glob("/usr/local/lib/python*/dist-packages/vllm/model_executor/layers/attention/attention.py") +
        glob.glob("/usr/local/lib/python*/site-packages/vllm/model_executor/layers/attention/attention.py")
    )

    patched = False
    for attn_file in attn_candidates:
        with open(attn_file, "r") as f:
            content = f.read()

        marker = "self.impl = impl_cls("
        pop_line = "extra_impl_args.pop('layer_idx', None)"
        if marker in content and pop_line not in content:
            content = content.replace(marker, pop_line + "\n        " + marker, 1)
            with open(attn_file, "w") as f:
                f.write(content)
            print(f"Patched: {attn_file}")
            patched = True
        else:
            print(f"Skipped (already patched, or marker not found): {attn_file}")

    # The shim wraps FlashAttention.__init__ so it silently swallows layer_idx.
    # 这个 shim 包装 FlashAttention.__init__，让它静默吞掉 layer_idx。
    sitecustomize_code = (
        "import sys\n"
        "def _patch():\n"
        "    import importlib\n"
        "    for mn in ['flash_attn','flash_attn.flash_attn_interface']:\n"
        "        try:\n"
        "            m=importlib.import_module(mn)\n"
        "            for a in dir(m):\n"
        "                if 'FlashAttention' in a:\n"
        "                    c=getattr(m,a)\n"
        "                    if hasattr(c,'__init__'):\n"
        "                        _o=c.__init__\n"
        "                        def _p(self,*aa,layer_idx=None,**kw):\n"
        "                            return _o(self,*aa,**kw)\n"
        "                        c.__init__=_p\n"
        "                        print(f'[sitecustomize] patched {mn}.{a}',file=sys.stderr)\n"
        "                        return\n"
        "        except Exception: pass\n"
        "_patch()\n"
    )
    with open("/kaggle/working/sitecustomize.py", "w") as f:
        f.write(sitecustomize_code)
    print("Created /kaggle/working/sitecustomize.py")

    if patched:
        print("Direct patch applied — layer_idx will be filtered out.")
    else:
        print("WARNING: direct patch not applied; relying on sitecustomize.py + PYTHONPATH")
else:
    print("flash-attn is available — no patch needed")

print("\n=== Install complete ===")

---


# 2. Launch the vLLM server

Starts the server in the background, logs to `/kaggle/working/vllm_server.log`, waits
for it to come up, verifies the attention backend, and fires one smoke request. The
kernel stays free while the model loads.


# 2. 启动 vLLM 服务

在后台启动服务，日志写到 `/kaggle/working/vllm_server.log`，等它起来，验证 attention
后端，然后发一条冒烟请求。模型加载期间内核不会被占用。


## The FP8 / attention-backend problem

This is the single most important thing in the notebook, because getting it wrong
produces a crash rather than a warning.

FlashInfer's FP8 attention kernels exist **only on SM90+ (Hopper and newer)**. The L4
is **SM89**. So with `--kv-cache-dtype fp8` enabled, the FlashInfer path reaches a
kernel that was never implemented for this architecture and dies with a **CUDA illegal
memory access** (vLLM issue #44879 / PR #45038).

Triton implements the same operation. Switching the backend to `TRITON_ATTN` keeps the
FP8 KV cache working — and with it, half the KV memory and roughly double the number of
tokens that fit.

**Setting the CLI flag is not enough.** The MTP drafter does **not** inherit
`--attention-backend`; it picks a backend on its own and chooses FlashInfer again. That
leaves the main model on Triton and the drafter on FlashInfer, which crashes just the
same (vLLM PR #53450). The `VLLM_FORCE_ATTN_BACKEND` environment variable is what pins
the drafter too — it is the one line that makes the fix stick.


## FP8 与 attention 后端的坑

这是整个 notebook 里最要紧的一点，因为配错的后果是崩溃而不是警告。

FlashInfer 的 FP8 attention kernel **只在 SM90+（Hopper 及以上）存在**。L4 是
**SM89**。所以开着 `--kv-cache-dtype fp8` 走 FlashInfer 会踩到一个从未在该架构上实现
的 kernel，直接报 **CUDA illegal memory access**（vLLM issue #44879 / PR #45038）。

Triton 有同样的实现。把后端换成 `TRITON_ATTN` 就能让 FP8 KV cache 正常工作 ——
也就保住了减半的 KV 显存，能装下的 token 数大约翻倍。

**只设 CLI 参数不够。** MTP 的 drafter **不继承** `--attention-backend`，
它会自己挑一个后端，然后又选回 FlashInfer。结果是主模型走 Triton、drafter 走
FlashInfer，照样崩（vLLM PR #53450）。`VLLM_FORCE_ATTN_BACKEND` 这个环境变量才是把
drafter 也钉住的那一下 —— 它是让整个修复生效的关键一行。


## The libcuda.so fix

Two copies of `libcuda.so` live on this image:

| Path | Version | Role |
|------|---------|------|
| `/usr/local/nvidia/lib64` | 580.178.04 | the real driver |
| `/usr/local/cuda-12.8/compat` | 570.124.06 | an older compat build |

vLLM must load the real one. Both search paths put that directory first, and
`LD_PRELOAD` forces it outright rather than leaving it to resolution order. If vLLM
fails with a driver or symbol error, this block is the first thing to check.


## libcuda.so 的修复

这个镜像上有两份 `libcuda.so`：

| 路径 | 版本 | 角色 |
|------|------|------|
| `/usr/local/nvidia/lib64` | 580.178.04 | 真正的驱动 |
| `/usr/local/cuda-12.8/compat` | 570.124.06 | 较旧的 compat 版本 |

vLLM 必须加载真的那份。两个搜索路径都把该目录放在最前面，并用 `LD_PRELOAD` 直接钉死，
而不是交给解析顺序去决定。如果 vLLM 报驱动或符号错误，先查这一段。


## Verifying the backend actually took effect

Passing a flag is not proof. vLLM logs the backend it selected, and the drafter is the
half most likely to have quietly kept FlashInfer — so the cell reads that line back and
warns if `FLASHINFER` still appears.

It then sends **one smoke request**. A crash from the FP8/MTP interaction surfaces
there, not at `/health`, and it surfaces in the log rather than in the HTTP response —
so the log is checked explicitly for `illegal memory access`, `EngineCore encountered a
fatal error`, and `EngineDeadError`.

Doing this in the launch cell is deliberate: the failure mode is a crash minutes or
hours into a run, and it is far cheaper to catch it here.


## 确认后端真的生效了

传了参数不等于生效。vLLM 会打印它实际选中的后端，而 drafter 正是最可能悄悄留在
FlashInfer 的那一半 —— 所以这个 cell 会把那行读回来，如果仍然出现 `FLASHINFER` 就告警。

然后它会发**一条冒烟请求**。FP8/MTP 交互导致的崩溃会在那里暴露，而不是在 `/health`；
而且它记在日志里而非 HTTP 响应里 —— 所以这里会显式检查日志中的
`illegal memory access`、`EngineCore encountered a fatal error` 和 `EngineDeadError`。

把这件事放在启动 cell 里是刻意的：这个故障的表现形式是跑到几分钟甚至几小时后崩溃，
在这里发现要便宜得多。


## Notes on the flags

- `NUM_SPEC_TOKENS = 6` is the speculative-decoding (MTP) token count. MTP ≥ 2 was the
  crash-prone region historically; drop to 2 if you hit one.
- Surviving startup does **not** mean the drafter is useful. The acceptance rate can
  still be 0%, which is a separate bug in the same issue. Its symptom is garbled or
  repetitive output, and the fix is to lower `NUM_SPEC_TOKENS` or drop
  `--speculative-config` entirely.
- `--gpu-memory-utilization 0.94` is the VRAM ceiling vLLM may claim. If you hit OOM,
  cut `--max-model-len` first — 131072 is far more context than ARC needs.
- `--max-num-seqs 32` caps concurrent sequences server-side. `n` in section 5 must not
  exceed it.
- To get a stack pointing at the actual offending kernel instead of an async-reporting
  false one, set `CUDA_LAUNCH_BLOCKING=1` and `TORCH_USE_CUDA_DSA=1` in `env`. It is
  much slower, so only do it while debugging.

## 关于这些参数

- `NUM_SPEC_TOKENS = 6` 是投机解码（MTP）的 token 数。历史上 MTP ≥ 2 是崩溃高发区；
  遇到崩溃就降到 2。
- 能启动**不**等于 drafter 有用。接受率仍可能是 0%，那是同一个 issue 里的另一个独立
  bug。它的症状是输出乱码或重复，处理办法是降低 `NUM_SPEC_TOKENS`，
  或者干脆去掉 `--speculative-config`。
- `--gpu-memory-utilization 0.94` 是允许 vLLM 占用的显存上限。如果 OOM，
  先砍 `--max-model-len` —— 131072 远超 ARC 所需。
- `--max-num-seqs 32` 限制服务端的并发序列数。第 5 节的 `n` 不能超过它。
- 想拿到指向真正肇事 kernel 的栈（而不是异步上报的假栈），在 `env` 里设置
  `CUDA_LAUNCH_BLOCKING=1` 和 `TORCH_USE_CUDA_DSA=1`。会明显变慢，只在排查时用。


In [ ]:
import os, subprocess, time, urllib.request, signal, sys, json, threading

model_path = "/kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1"
log_file = "/kaggle/working/vllm_server.log"

# --- Attention backend: Triton, not FlashInfer ---
# FlashInfer's FP8 attention kernels exist only on SM90+ (Hopper). The L4 is SM89, so
# with --kv-cache-dtype fp8 the FlashInfer path reaches an unimplemented kernel and dies
# with a CUDA illegal memory access (vLLM #44879 / PR #45038). Triton implements it, so
# switching the backend keeps the FP8 KV cache — and with it double the KV capacity.
# FlashInfer 的 FP8 attention kernel 只在 SM90+（Hopper 及以上）存在。L4 是 SM89，
# 于是开着 --kv-cache-dtype fp8 走 FlashInfer 会踩到未实现的那条路，
# 报 CUDA illegal memory access（vLLM #44879 / PR #45038）。Triton 有实现，
# 换后端就能保住 FP8 KV cache —— 也就保住了 2 倍的 KV 容量。
#
# Setting the CLI flag alone is NOT enough: the MTP drafter does not inherit
# --attention-backend and picks FlashInfer again by itself, so the main model runs
# Triton while the drafter runs FlashInfer and it still crashes (vLLM PR #53450).
# The VLLM_FORCE_ATTN_BACKEND env var below is what pins the drafter too.
# 只设 CLI 参数不够：MTP 的 drafter 不继承 --attention-backend，会自己选回 FlashInfer，
# 结果是主模型 Triton + drafter FlashInfer，照样崩（vLLM PR #53450）。
# 下面的 VLLM_FORCE_ATTN_BACKEND 才是把 drafter 也钉住的那一下。
ATTN_BACKEND = "TRITON_ATTN"

# Speculative decoding (MTP) token count. 6 works at this configuration; drop to 2 if
# you hit a crash — MTP >= 2 was the crash-prone region. Note that surviving startup
# does not mean the drafter is useful: the acceptance rate can still be 0%, which is a
# separate bug in the same issue (#44879).
# 投机解码（MTP）的 token 数。当前配置下 6 可用；遇到崩溃就降到 2 ——
# MTP >= 2 曾是崩溃高发区。注意能启动不等于 drafter 有用：
# 接受率仍可能是 0%，那是同一个 issue（#44879）里的另一个独立 bug。
NUM_SPEC_TOKENS = 6

cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", model_path,
    "--served-model-name", "qwen3.8-27b",
    "--host", "0.0.0.0", "--port", "8000",
    # Weights are sharded across all four GPUs; 0.94 is the VRAM ceiling vLLM may claim.
    # 权重切分到四张卡上；0.94 是允许 vLLM 占用的显存上限。
    "--tensor-parallel-size", "4",
    "--gpu-memory-utilization", "0.96",
    # Far more context than ARC needs. Headroom, not a target — cut it first on OOM.
    # 远超 ARC 实际所需的上下文。属于预留余量而非目标值 —— OOM 时第一个砍它。
    "--max-model-len", "131072",
    "--max-num-seqs", "16",
    "--trust-remote-code",
    "--default-chat-template-kwargs", '{"reasoning_effort": "medium"}',
    # fp8 KV cache halves KV memory, doubling how many tokens fit — this is the whole
    # reason the attention backend has to be Triton (see above).
    # fp8 KV cache 把 KV 显存减半，能装下的 token 翻倍 ——
    # 这正是 attention 后端必须换成 Triton 的原因（见上方）。
    "--kv-cache-dtype", "fp8",
    "--attention-backend", ATTN_BACKEND,
    # Qwen3's native tool-call and reasoning-tag parsing. The pipeline does not use tool
    # calls, but the reasoning parser is what lets chat() read the thinking trace.
    # 启用 Qwen3 原生的 tool-call 与 reasoning 标签解析。流水线不用 tool call，
    # 但 reasoning parser 是 chat() 能把思考过程读回来的前提。
    "--enable-auto-tool-choice",
    "--no-async-scheduling",
    "--tool-call-parser", "qwen3_coder",
    "--reasoning-parser", "qwen3",
    "--speculative-config", json.dumps(
        {"method": "mtp", "num_speculative_tokens": NUM_SPEC_TOKENS}),
]

env = os.environ.copy()
env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
env["VLLM_LOGGING_LEVEL"] = "INFO"
env["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"
env["PYTHONPATH"] = "/kaggle/working" + os.pathsep + env.get("PYTHONPATH", "")
# The one line that also pins the MTP drafter to Triton. See the note above.
# 把 MTP 的 drafter 也钉到 Triton 的那一行。见上面的说明。
env["VLLM_FORCE_ATTN_BACKEND"] = ATTN_BACKEND

# --- libcuda.so: put the real driver first and preload it ---
# Two copies of libcuda.so are on this image: the real driver (580.178.04, in
# /usr/local/nvidia/lib64) and an older compat build (570.124.06, in
# /usr/local/cuda-12.8/compat). vLLM must load the real one, so that directory goes
# first on both search paths and LD_PRELOAD forces it outright.
# 这个镜像上有两份 libcuda.so：真驱动（580.178.04，在 /usr/local/nvidia/lib64）
# 和一份较旧的 compat 版本（570.124.06，在 /usr/local/cuda-12.8/compat）。
# vLLM 必须加载真的那份，所以把该目录放在两个搜索路径的最前面，
# 并用 LD_PRELOAD 直接钉死。
cuda_real_lib   = "/usr/local/nvidia/lib64"
cuda_compat_lib = "/usr/local/cuda-12.8/compat"

env["LIBRARY_PATH"] = (
    cuda_real_lib + ":" + cuda_compat_lib + ":" + env.get("LIBRARY_PATH", "")
)
env["LD_LIBRARY_PATH"] = (
    cuda_real_lib + ":" + cuda_compat_lib + ":" + env.get("LD_LIBRARY_PATH", "")
)
env["LD_PRELOAD"] = cuda_real_lib + "/libcuda.so"

print(f"\n=== Launching vLLM TP=4, len=131K, seqs=32, util=0.94, "
      f"MTP={NUM_SPEC_TOKENS}, attn={ATTN_BACKEND} ===")
print(f"LIBRARY_PATH   = {env['LIBRARY_PATH']}")
print(f"LD_LIBRARY_PATH= {env['LD_LIBRARY_PATH']}")
print(f"LD_PRELOAD     = {env['LD_PRELOAD']}")
print(f"VLLM_FORCE_ATTN_BACKEND = {env['VLLM_FORCE_ATTN_BACKEND']}")

# start_new_session puts the server in its own process group, so the failure path
# below can kill the whole group rather than just the parent process.
# start_new_session 让服务进入独立进程组，这样下面的失败分支可以杀掉整个组，
# 而不只是父进程。
with open(log_file, "w") as fh:
    proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL, text=True, start_new_session=True,
        env=env)


def read_log():
    if not os.path.exists(log_file):
        return []
    with open(log_file, errors="ignore") as f:
        return f.readlines()


# Poll /health for up to 12 minutes. Loading the weights and capturing CUDA graphs
# takes several minutes; giving up early is the most common false alarm.
# 轮询 /health，最多 12 分钟。加载权重和捕获 CUDA graph 要花好几分钟，
# 过早放弃是最常见的误报来源。
started = False
for _ in range(360):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=3):
            print("✅ vLLM started successfully!")
            subprocess.run("nvidia-smi", shell=True)
            started = True
            break
    except Exception:
        if proc.poll() is not None:
            break
        time.sleep(2)

# On failure, surface the real cause from the log rather than a bare timeout.
# 失败时从日志里找出真正的原因，而不是只报一个超时。
if not started:
    print("❌ Startup failed")
    lines = read_log()
    for kw in ["OOM", "OutOfMemory", "CUDA out of memory", "ValueError",
               "RuntimeError", "ERROR", "error:", "unrecognized arguments",
               "illegal memory access"]:
        for l in lines:
            if kw in l:
                print(f"  [{kw}]: {l.rstrip()[:200]}")
    print("\nLast 30 log lines:")
    for l in lines[-30:]:
        print(f"  {l.rstrip()[:200]}")
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    except Exception:
        pass
    raise RuntimeError("vLLM failed to start")

# --- Verify the backend that actually took effect ---
# Passing the flag is not proof: vLLM logs the backend it selected, and the drafter is
# the part most likely to have silently kept FlashInfer.
# 光传了参数不算数：vLLM 会打印它实际选中的后端，
# 而 drafter 正是最可能悄悄留在 FlashInfer 的那一半。
print("\n=== Checking the attention backend actually in use ===")
lines = read_log()
backend_lines = [l.rstrip() for l in lines if "attention backend" in l.lower()]
if backend_lines:
    for l in backend_lines:
        print(f"  {l[:200]}")
    if any("TRITON_ATTN" in l.upper() for l in backend_lines):
        print("✅ Found TRITON_ATTN")
    if any("FLASHINFER" in l.upper() for l in backend_lines):
        print("⚠️  FLASHINFER is still present — the drafter was not pinned by "
              "VLLM_FORCE_ATTN_BACKEND, and this will most likely crash")
else:
    print("⚠️  No 'attention backend' line in the log; check the log manually")

# --- Smoke test: one request ---
# A crash from the FP8/MTP interaction surfaces here, not at /health.
# FP8/MTP 交互导致的崩溃会在这里暴露，而不是在 /health 那一步。
print("\n⏳ Smoke test: one request...")
try:
    req = urllib.request.Request(
        "http://127.0.0.1:8000/v1/chat/completions",
        data=json.dumps({
            "model": "qwen3.8-27b",
            "messages": [{"role": "user", "content": "hi"}],
            "max_tokens": 16,
            "temperature": 0.0,
        }).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=900) as r:
        print(f"✅ Smoke test passed (HTTP {r.status})")
except Exception as e:
    print(f"❌ Smoke test failed: {str(e)[:200]}")

# A fatal crash lands in the log rather than in the HTTP response, so check it directly.
# 致命崩溃会记在日志里而不是 HTTP 响应里，所以单独查一次。
fatal = [l.rstrip() for l in read_log()
         if "illegal memory access" in l.lower()
         or "EngineCore encountered a fatal error" in l
         or "EngineDeadError" in l]
if fatal:
    print("\n⚠️  Fatal error found in the log:")
    for l in fatal[:10]:
        print(f"   {l[:200]}")
    print("   Next step: lower NUM_SPEC_TOKENS to 2, or drop --kv-cache-dtype fp8.")
else:
    print("✅ No fatal error in the log — the server is healthy.")


---


# 3. Wait for readiness

A readiness gate. `/health` returns 200 as soon as the process is listening, but the
model may still be loading — `/v1/models` only answers once the engine is genuinely
serving. This cell polls that endpoint and prints the model IDs it advertises.

That matters because section 4's `chat()` sends `"model": "qwen3.8-27b"`, which must
match the `--served-model-name` from section 2. If you rename the served model, change
it in both places.

Safe to re-run: if the server is already up, this returns immediately.

> If this times out, section 2 did not succeed — scroll up and read
> `/kaggle/working/vllm_server.log`.

# 3. 等待就绪

一道就绪闸门。进程一监听端口 `/health` 就返回 200，但此时模型可能还在加载 ——
只有引擎真正开始服务，`/v1/models` 才会应答。这个 cell 轮询该端点，
并打印它公布的模型 ID。

这一步有意义，是因为第 4 节 `chat()` 发送的 `"model": "qwen3.8-27b"` 必须和
第 2 节的 `--served-model-name` 一致。如果你改了 serving 的名字，两处都要改。

可以安全重跑：服务已在运行时这个 cell 会立刻返回。

> 如果这里超时，说明第 2 节没成功 —— 往上翻，读
> `/kaggle/working/vllm_server.log`。


In [ ]:
import time, json, urllib.request

# /health answers as soon as the process is listening, but the model may still be
# loading. /v1/models only answers once the engine is genuinely serving, so that is
# the endpoint to gate on.
# /health 在进程一监听端口时就应答，但模型可能还在加载。
# /v1/models 只有引擎真正开始服务后才会应答，所以以它为准。
BASE = "http://127.0.0.1:8000"

def healthy():
    try:
        with urllib.request.urlopen(BASE + "/v1/models", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

# Wait up to 90 minutes, printing a heartbeat once a minute.
# 最多等 90 分钟，每分钟打印一次心跳。
t0 = time.time()
while time.time() - t0 < 5400:
    if healthy():
        break
    el = int(time.time() - t0)
    if el and el % 60 < 3:
        print(f"... server starting, {el//60} min elapsed")
    time.sleep(5)

# Report the served model IDs. The pipeline's chat() must send one of these; it is
# hardcoded to match --served-model-name from the previous cell.
# 打印服务公布的模型 ID。流水线的 chat() 必须发送其中之一；
# 它是写死的，要和上一个 cell 的 --served-model-name 保持一致。
if healthy():
    try:
        with urllib.request.urlopen(BASE + "/v1/models", timeout=5) as r:
            models = json.load(r)["data"]
        print("✅ vLLM ready. Available models:", [m["id"] for m in models])
    except Exception as e:
        print("vLLM is ready, but reading the model list failed:", e)
else:
    print("❌ vLLM is not ready — run the 'Launch the vLLM server' cell above first")

print(f"The pipeline will call: {BASE}/v1/chat/completions   (model=qwen3.8-27b, no auth)")


---


# 4. The pipeline

This is the substance of the project, and it is written as a self-contained module: it
has its own `argparse` entry point and a `__main__` guard, so you can copy this one cell
out into `arc_pipeline.py` and run it outside Kaggle unchanged. It is inlined here so
the notebook works as a single artifact.

The code cell is divided into ten labelled sections. Each is described below.


# 4. 流水线

这是项目的主体，而且它是写成一个自包含模块的：自带 `argparse` 入口和 `__main__` 守卫，
所以你可以把这一个 cell 单独拷成 `arc_pipeline.py`，脱离 Kaggle 直接运行。
内联在这里是为了让整个 notebook 作为单一产物可用。

代码 cell 分成十个带标签的小节，下面逐一说明。


## Data flow

```
 challenges.json
       │
       ▼
 ┌─────────────┐   one prompt per task (shortest first)
 │ build_prompt│──────────────────────────────┐
 └─────────────┘                              │
                                              ▼
                              ┌──────────────────────────────┐
                              │ solve_challenges             │
                              │  ThreadPoolExecutor(n=32)    │
                              │  rolling refill + deadline   │
                              └──────────────┬───────────────┘
                                             │  chat(prompt)
                                             ▼
                                    ┌─────────────────┐
                                    │  vLLM :8000     │
                                    └────────┬────────┘
                                             │  raw text / JSON
                                             ▼
                              ┌──────────────────────────────┐
                              │ normalize_response           │
                              │  extract → validate → slots  │
                              └──────────────┬───────────────┘
                                             ▼
                              output/result_<taskid>.json  (one per task)
                                             │
                                             ▼
                    aggregate_to_submission  ← _fallback_grids for anything missing
                                             │
                                             ▼
                                   submission.json   (always complete)
                                             │
                                     ┌───────┴────────┐
                                     ▼                ▼
                              score_submission   (upload to Kaggle)
                                     │
                                     ▼
                                 score.json
```


## 数据流

```
 challenges.json
       │
       ▼
 ┌─────────────┐   每个任务一个 prompt（短的在前）
 │ build_prompt│──────────────────────────────┐
 └─────────────┘                              │
                                              ▼
                              ┌──────────────────────────────┐
                              │ solve_challenges             │
                              │  ThreadPoolExecutor(n=32)    │
                              │  滚动补位 + 整体时限          │
                              └──────────────┬───────────────┘
                                             │  chat(prompt)
                                             ▼
                                    ┌─────────────────┐
                                    │  vLLM :8000     │
                                    └────────┬────────┘
                                             │  原始文本 / JSON
                                             ▼
                              ┌──────────────────────────────┐
                              │ normalize_response           │
                              │  提取 → 校验 → 装槽位         │
                              └──────────────┬───────────────┘
                                             ▼
                              output/result_<taskid>.json  (每任务一个)
                                             │
                                             ▼
                    aggregate_to_submission  ← 缺失的由 _fallback_grids 补齐
                                             │
                                             ▼
                                   submission.json   （永远完整）
                                             │
                                     ┌───────┴────────┐
                                     ▼                ▼
                              score_submission   (上传到 Kaggle)
                                     │
                                     ▼
                                 score.json
```


## Part by part


## 逐节说明


### 1) Model client

`chat()` is the only place that talks to the model. It POSTs one prompt and returns the
assistant's text, with `temperature=0.0` and `enable_thinking=True`.

Its timeout is not a fixed number: `_chat_timeout()` clamps it to whatever remains of
the overall deadline, so a request started near the end cannot outlive the run.

Two details are easy to get wrong and are handled here:

- **Reasoning tags.** vLLM versions differ on where the thinking trace lands —
  `reasoning`, `reasoning_content`, or still inline in `content`. All three are tried.
- **The prompt log.** Every call appends the prompt and reasoning trace to
  `thinking.log`, which is your only window into *why* the model produced a wrong grid.


### 1) 模型客户端

`chat()` 是唯一与模型通信的地方。它发送一个 prompt，返回 assistant 的文本，
`temperature=0.0`、`enable_thinking=True`。

它的超时不是固定值：`_chat_timeout()` 会把它压到整体剩余时限之内，
所以临近截止时发起的请求不会拖过整轮时限。

有两个细节容易踩坑，这里替你处理了：

- **推理标签。** 不同 vLLM 版本把思考过程放在不同位置 —— `reasoning`、
  `reasoning_content`，或者仍留在 `content` 里。这里三种都试。
- **prompt 日志。** 每次调用都把 prompt 和推理过程追加到 `thinking.log`，
  这是你唯一能看出模型**为什么**给出错误网格的窗口。


### 2) Configuration and the deadline

Every knob in one block, plus the clock the whole run is bounded by.

`RUN_START_TS` defaults to the moment this cell runs, but a notebook can set it in an
earlier cell so the budget is measured from the real start. `DEADLINE_MIN` defaults to
690 minutes (11h30m). **Set this below your session limit** — the deadline only
helps if it fires before the platform kills the process.

`seconds_left()` is what everything else consults. `_chat_timeout()` uses it to clamp
per-request timeouts; section 7 uses it to decide whether to start a task at all.


### 2) 配置与整体时限

所有旋钮集中在一块，外加约束整轮运行的时钟。

`RUN_START_TS` 默认取这个 cell 运行的时刻，但 notebook 可以在更早的 cell 里设置它，
让预算从真正的起点开始计。`DEADLINE_MIN` 默认 690 分钟（11 小时 30 分）——
**要设得低于你的会话上限**，否则平台会在时限触发之前就把进程杀掉。

`seconds_left()` 是其他一切都要咨询的东西：`_chat_timeout()` 用它压单次请求的超时，
第 7 节用它决定一个任务到底要不要启动。


### 3) JSON and grid helpers

`load_json` / `save_json` are thin wrappers. `_fmt_sec` formats durations.

`_normalize_grid` is the strict one: it coerces anything that might be a grid into a
rectangular 2-D list of ints and returns `None` otherwise. Rejecting malformed grids
here rather than submitting them matters — Kaggle's scorer rejects an *entire* malformed
submission instead of scoring the rest.


### 3) JSON 与网格工具

`load_json` / `save_json` 是薄封装。`_fmt_sec` 把时长格式化。

`_normalize_grid` 是严格的那一个：把任何可能是网格的东西规整成矩形的二维 int 列表，
否则返回 `None`。在这里拒绝非法网格而不是提交出去很重要 —— Kaggle 的打分器会拒收
**整份**非法提交，而不是只扣那一部分。


### 4) Prompt construction

`convert_payload_to_text` renders a task into the flat line format the model was tuned
on: a literal marker (`train` / `input` / `output` / `test`) followed by each grid row
as a run of digits — `0123` rather than `[0,1,2,3]`. Every test input gets its own
`input` block, in order, so a three-input task produces three.

`build_prompt` then appends the instruction block, which spells out the attempt-key
contract with a worked example, and a closing instruction worth noting: **wide rows must
use `|` separators every four cells**. Without it the model miscounts columns on grids
wider than eight cells and emits ragged rows, which `_normalize_grid` then rejects —
a pure formatting loss, not a reasoning failure.

> The instructions are **in Chinese, deliberately**. The reported scores were produced
> with exactly this prompt. Translating it changes model behaviour, so re-run the eval
> before trusting the number if you do.


### 4) Prompt 构造

`convert_payload_to_text` 把任务渲染成模型训练时使用的扁平行格式：先是字面标记
（`train` / `input` / `output` / `test`），然后每个网格行是一串数字 ——
`0123` 而不是 `[0,1,2,3]`。每个 test 输入都会产出自己独立的 `input` 块并按顺序排列，
所以三个输入的任务会产出三段。

`build_prompt` 接着追加指令块，写明 attempt 键的契约并给出例子；结尾还有一条值得注意的
指令：**宽度超过 8 格的网格，每 4 个元素要用 `|` 分隔**。没有它的话，
模型会数错列数、输出参差不齐的行，然后被 `_normalize_grid` 拒掉 ——
那是纯格式损失，不是推理失败。

> 指令是**中文的，而且是刻意的**。报告出来的分数就是用这份 prompt 跑出来的。
> 翻译它会改变模型行为，所以如果你翻译了，请先重跑评测再相信那个数字。


### 5) Response normalization

The model's output is not trustworthy. In practice it comes back as JSON wrapped in
prose, inside ```` ```json ```` fences, as Python single-quoted literals, as a bare list
instead of a dict, or nested one level deeper than expected. This section absorbs all
of that:

- `_extract_from_text` digs a JSON object out of arbitrary surrounding text, falling
  back to `ast.literal_eval`.
- `_parse_attempt_index` turns `"attempt_3"` into index `2`.
- `_container_to_slots` maps whatever container shape arrived onto `k` fixed slots.
- `_normalize_grid` (section 3) validates each grid.

The contract everything downstream relies on: **`normalize_response` returns exactly
`max(2, n_outputs)` slots**, each either a valid grid or `None`. A partial answer is
kept — slots the model did not fill stay `None` rather than failing the whole task.


### 5) 响应归一化

模型的输出不可信。实践中它会以被散文包住的 JSON、被 ```` ```json ```` 围栏包住的
JSON、Python 单引号字面量、裸 list 而不是 dict、或者比预期多套一层的形式返回。
这一节把这些全部吸收掉：

- `_extract_from_text` 从任意前后文中挖出 JSON 对象，回退到 `ast.literal_eval`。
- `_parse_attempt_index` 把 `"attempt_3"` 转成下标 `2`。
- `_container_to_slots` 把无论什么形状的容器映射到固定的 `k` 个槽位。
- `_normalize_grid`（第 3 节）校验每个网格。

下游一切依赖的契约是：**`normalize_response` 恰好返回 `max(2, 输出个数)` 个槽位**，
每个要么是合法网格，要么是 `None`。部分答案会被保留 ——
模型没填出来的槽位保持 `None`，而不是让整个任务失败。


### 6) Logging and failure diagnosis

`_append_log` writes one timestamped line per event to `output/solve.log`; `_save_raw`
keeps the raw model output whenever parsing fails. Both swallow their own errors — a
logging failure must never kill a run.

`_classify_error` turns an exception into a plain-English cause, separating transport
failures from everything else. `_why_no_grid` distinguishes "the model only emitted
thinking and never produced JSON" — usually a `max_tokens` truncation — from "there was
JSON but the keys or shapes were wrong", and prints the offending keys with their types.


### 6) 日志与失败诊断

`_append_log` 每个事件往 `output/solve.log` 写一行带时间戳的记录；
`_save_raw` 在解析失败时保留模型原始输出。两者都吞掉自身的错误 ——
日志失败绝不能中断整个运行。

`_classify_error` 把异常翻译成人话原因，把传输层故障和其他问题分开。
`_why_no_grid` 区分"模型只输出了思考、始终没给出 JSON"（通常是 `max_tokens` 被截断）
和"有 JSON 但键或形状不对"，并打印出有问题的键及其类型。


### 7) Parallel solving under the deadline

`_solve_one` handles a single task: build prompt → call `chat` → normalize → write
`output/result_<taskid>.json`. On failure it writes **no** result file, which is what
makes `SKIP_EXISTING` a correct resume mechanism and what tells aggregation the task
needs a fallback.

`solve_challenges` sorts tasks by **prompt length, shortest first**, and submits them to
a `ThreadPoolExecutor`. Its queue is FIFO, so short prompts start first, and because the
pool is rolling rather than batched, no slot idles waiting for the slowest member of a
batch. Generation length varies by more than an order of magnitude across ARC tasks, so
this matters a great deal for wall-clock time.

Working against that is the deadline. Two places check it:

- **Before starting a task.** Once `seconds_left()` is non-positive, queued tasks are
  recorded as `deadline` and never started.
- **Inside `_solve_one`.** A task that is already running stops before opening another
  request.

So the run degrades in a controlled way rather than being killed mid-write. Note that
`MAX_RETRIES` is 0 by default: with `temperature=0.0` a retry reproduces the same
answer, so it buys little and costs time you may not have.

A background thread prints progress with a running mean and an ETA.


### 7) 并行求解，受整体时限约束

`_solve_one` 处理单个任务：构造 prompt → 调 `chat` → 归一化 → 写
`output/result_<taskid>.json`。失败时它**不写**结果文件 ——
这既是 `SKIP_EXISTING` 能作为正确断点续跑机制的原因，
也是汇总阶段据此判断该任务需要兜底的依据。

`solve_challenges` 按 **prompt 长度从短到长**排序，提交给 `ThreadPoolExecutor`。
它的队列是 FIFO，所以短 prompt 先开始；又因为这是滚动池而非分批，
没有任何槽位会为了等一批里最慢的那个而空转。ARC 任务的生成长度相差一个数量级以上，
这对墙钟时间影响很大。

与之制衡的是整体时限，有两处检查它：

- **启动任务之前。** 一旦 `seconds_left()` 不为正，排队中的任务被记为 `deadline`，
  永远不会启动。
- **`_solve_one` 内部。** 已经在跑的任务在开下一个请求之前会停下。

于是整轮运行是可控地降级，而不是在写文件写到一半时被强杀。注意 `MAX_RETRIES`
默认是 0：`temperature=0.0` 下重试会复现同一个答案，所以收益很小，
却一定要花你可能没有的时间。

一个后台线程打印进度，带滑动均值和预计剩余时间。


### 8) Aggregation and the submission guarantee

`aggregate_to_submission` collects `output/result_*.json` into a single
`submission.json`. It is the only thing standing between a crashed run and an unscorable
file, so it never emits a `null`.

For every task it starts from a complete fallback entry (`_fallback_entry`) and then
overwrites the slots holding a real answer. A task whose result file is missing,
unreadable, truncated mid-write, or only partially valid therefore degrades to its
fallback grid instead of producing a `null`.

The fallback entry comes from `_fallback_grids`, which guesses from patterns visible in
that task's own `train` pairs, with no model call:

1. every `train` output is identical → a constant task, reuse that grid;
2. some `train` output equals its own input → identity-like, reuse the test input;
3. otherwise → the most common `train` output.

For a 1-output task the second slot is given a *different* candidate, which buys the
free second guess that scoring allows there. If even that yields nothing usable — a task
with no test input, say — `_placeholder_grid` supplies an all-zero grid as the last
resort: wrong, but structurally valid, which is all the format requires.

Finally `ensure_submission_complete` walks the whole submission and repairs any slot that
is still not a valid grid. It returns the number of slots it fixed; the `null` count
printed next to it should be 0.

It reads files only — which is why it is free to re-run after a scoring or normalization
fix, with no model calls.


### 8) 汇总与提交保障

`aggregate_to_submission` 把 `output/result_*.json` 收拢成单个 `submission.json`。
它是"运行崩了"和"文件没法计分"之间唯一的一道墙，所以它绝不会写出 `null`。

对每个任务，它都从一个完整的兜底条目（`_fallback_entry`）开始，
再覆盖确实有答案的槽位。于是结果文件缺失、读不出、写到一半被截断、或只有部分合法的任务，
都会退化成它的兜底网格，而不会产生 `null`。

兜底条目来自 `_fallback_grids`，只用该任务自己 `train` 里可见的规律猜答案，不调用模型：

1. `train` 的输出全都一样 → 常数任务，直接用那个网格；
2. 有 `train` 的输出等于自己的输入 → 类恒等任务，用对应的 test 输入；
3. 其余 → 用出现次数最多的那个 `train` 输出。

对 1 个输出的任务，第二个槽位会给一个**不同**的候选，
用上打分规则在这里允许的那次免费重猜。如果连这些都产不出可用网格 ——
比如没有 test 输入的任务 —— 就由 `_placeholder_grid` 提供全 0 网格作为最后兜底：
必然错误，但结构合法，而格式只要求合法。

最后 `ensure_submission_complete` 遍历整份提交，修复任何仍然不是合法网格的槽位。
它返回修复的槽位数；与它一起打印的 `null` 计数应该是 0。

它只读文件，所以修了打分或归一化之后可以随便重跑，不需要任何模型调用。


### 9) Scoring

`score_submission` compares `submission.json` against the solutions file. The formula is
spelled out below. `score_existing` prints the report and writes `score.json`.


### 9) 打分

`score_submission` 把 `submission.json` 和标准答案比对，公式见下。
`score_existing` 打印报告并写出 `score.json`。


### 10) Entry points

`run()` runs the guarantee in order — **pre-fill → solve → rebuild + repair**:

1. aggregate first, so a complete submission exists on disk before any model call;
2. solve, with each success overwriting its own slots in place;
3. aggregate again, which rebuilds from the result files and runs the repair pass.

Step 3 is reached whether the run finished normally or was cut off by the deadline.
`run()` prints the deadline window before starting. `main()` is the `argparse` CLI for
standalone use. The `__main__` guard
checks for an IPython kernel first, because in a notebook `sys.argv` holds the kernel's
own arguments rather than yours.


### 10) 入口

`run()` 按顺序执行整套保障 —— **预填充 → 求解 → 重建 + 修复**：

1. 先汇总一次，于是在任何模型调用之前，磁盘上就已经有一份完整的提交；
2. 求解，每成功一个就就地覆盖它自己的槽位；
3. 再汇总一次，从结果文件重建并跑修复兜底。

无论正常跑完还是被截止时间掐断，第 3 步都会走到。
`run()` 在开跑前会打印时限窗口。`main()` 是独立运行时的 `argparse` CLI。`__main__` 守卫会先检测 IPython 内核，
因为在 notebook 里 `sys.argv` 装的是内核自己的参数，不是你的。


## Attempt keys and scoring

This is the part most likely to be misread, so it is spelled out in full.

A task requires as many outputs as it has `test` inputs. The number of `attempt_*` keys
you must supply is `max(2, n_outputs)`:

| Test inputs | Required keys | How they are matched |
|-------------|---------------|----------------------|
| 1 | `attempt_1`, `attempt_2` | **Both are candidates for the same single output.** Either one matching scores the point. |
| 2 | `attempt_1`, `attempt_2` | `attempt_i` must match output `i`. Off-by-one scores nothing. |
| 3 | `attempt_1`, `attempt_2`, `attempt_3` | Same one-to-one rule. |

So with one output you get a free second guess; with several, the keys are positional and
that allowance disappears. This is exactly why the fallback deliberately gives a
1-output task two *different* candidates.

The score is 100 points split evenly across *outputs*, not tasks:

```
per_output = 100 / total_required_outputs      # 172 for this evaluation set
score      = per_output × (outputs predicted correctly)
```

A 3-output task is therefore worth three times a 1-output task.


## Attempt 键与打分

这是最容易被误读的部分，所以完整写清楚。

一个任务需要预测的输出个数，等于它在 `test` 里的输入个数。你必须给出的 `attempt_*`
键个数是 `max(2, 输出个数)`：

| test 输入数 | 必须给的键 | 匹配方式 |
|-------------|-----------|---------|
| 1 | `attempt_1`, `attempt_2` | **两个都是同一个唯一输出的候选。** 任意一个命中即得分。 |
| 2 | `attempt_1`, `attempt_2` | `attempt_i` 必须匹配第 i 个输出。错位不记分。 |
| 3 | `attempt_1`, `attempt_2`, `attempt_3` | 同上，一一对应。 |

也就是说：只有一个输出时你白得一次重猜机会；有多个输出时，键是位置性的，
额外重猜的机会就没了。这正是兜底逻辑刻意给 1 输出任务两个**不同**候选的原因。

满分 100 均分到**输出**上，不是任务：

```
per_output = 100 / total_required_outputs      # 本评测集为 172
score      = per_output × 预测正确的输出个数
```

因此一个 3 输出的任务，权重是一个 1 输出任务的 3 倍。


## Why this notebook is safe to cut off

Three mechanisms combine so that being interrupted is not a failure mode:

1. **The deadline**, set below the session limit, stops new work before the platform
   kills the process mid-write.
2. **Per-request timeouts** are clamped to the remaining budget, so nothing can hang past
   the deadline on a socket.
3. **The submission guarantee** keeps `submission.json` complete at every moment —
   written in full before inference starts, updated in place as answers arrive, and
   repaired slot by slot at the end. Even a result file truncated by a hard kill
   degrades to a fallback grid instead of crashing the aggregation.

The consequence: `submission.json` is always structurally complete and submittable, no
matter when the run ends. It also means the score reported after a truncated run is a
*lower* bound — the fallback grids score only by luck, which is the intended trade.


## 为什么这个 notebook 可以被安全掐断

三个机制配合，使得"被中断"不构成一种失败模式：

1. **整体时限**（要设得低于会话上限）在平台杀进程之前就停止新工作，
   而不是让它被杀在写文件写到一半的时候。
2. **单次请求超时**被压到剩余预算之内，所以没有请求能卡在 socket 上越过截止时刻。
3. **提交保障**让 `submission.json` 在任何时刻都是完整的 ——
   推理开始前就完整写出，随答案产出就地更新，结束时逐槽位修复兜底。
   即使结果文件被硬杀截断，也只会退化成兜底网格，而不会让整个汇总崩溃。

结果是：无论运行何时结束，`submission.json` 在结构上永远完整、永远可以提交。
这也意味着被截断后报出的分数是一个**下界** ——
兜底网格只能靠运气得分，而这正是刻意的取舍。


## Developer notes

- `chat_template_kwargs` is sent as a **top-level** field. vLLM and SGLang treat
  unrecognized top-level keys as `extra_body`, so this works on both — but it is not the
  OpenAI spec, and a strict proxy would reject it.
- `preserve_thinking=True` keeps the reasoning trace in the conversation history. It is
  irrelevant for single-turn solving and only matters if you extend this into a
  multi-turn agent.
- `thinking.log` is written without a lock. Under high `n` the lines from concurrent
  requests can interleave. It is a diagnostic file only, so this affects readability,
  not correctness — add a `threading.Lock()` around the write if you care.

## 开发者备注

- `chat_template_kwargs` 是作为**顶层字段**发送的。vLLM 和 SGLang 会把无法识别的
  顶层键当作 `extra_body`，所以在两者上都能用 —— 但这不是 OpenAI 规范，
  严格的代理会拒收。
- `preserve_thinking=True` 会把推理过程留在对话历史里。对单轮求解它没有影响，
  只有当你把它扩展成多轮 agent 时才有意义。
- `thinking.log` 的写入没有加锁。在 `n` 较大时，并发请求的日志行可能交错。
  它只是诊断文件，所以影响的是可读性而非正确性 —— 介意的话在写入外面加一个
  `threading.Lock()`。


In [ ]:
import collections
import json
import os
import re
import time
import threading
import urllib.request
from datetime import datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed


# ===========================================================================
# 1) Model client — the only place that talks to the served model
# 1) 模型客户端 —— 唯一与服务端模型通信的地方
# ===========================================================================

def chat(prompt, max_tokens=120000, api_key=""):
    # POST a single prompt and return the assistant's text. temperature=0.0 keeps
    # answers reproducible.
    # The timeout is not a fixed number: _chat_timeout() clamps it to whatever is left
    # of the overall deadline, so a request started near the end cannot outlive the run.
    # 发送单个 prompt，返回 assistant 的文本。temperature=0.0 保证结果可复现。
    # 超时不是固定值：_chat_timeout() 会把它压到整体剩余时限之内，
    # 于是临近截止时发起的请求不会拖过整轮时限。
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = "Bearer " + api_key

    body = {
        "model": "qwen3.8-27b",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        # vLLM and SGLang treat unrecognized top-level keys as extra_body, so the
        # chat-template options go here rather than nested under extra_body.
        # vLLM 和 SGLang 会把无法识别的顶层键当作 extra_body，
        # 所以 chat template 的选项放在这里，而不是嵌在 extra_body 里。
        "chat_template_kwargs": {"enable_thinking": True, "preserve_thinking": True},
    }

    req = urllib.request.Request(
        "http://127.0.0.1:8000/v1/chat/completions",
        data=json.dumps(body).encode(), headers=headers)
    with urllib.request.urlopen(req, timeout=_chat_timeout()) as r:
        response = json.load(r)

    msg = response["choices"][0]["message"]

    content = msg.get("content") or ""
    # Recent vLLM puts the reasoning trace in "reasoning"; older versions and the
    # Qwen gateway use "reasoning_content". Accept either.
    # 新版 vLLM 把推理过程放在 "reasoning"；旧版和 Qwen 网关用 "reasoning_content"。
    # 两者都兼容。
    reasoning = msg.get("reasoning") or msg.get("reasoning_content") or ""

    # Last resort: if the server did not split the fields and the thinking is still
    # embedded in content, peel it out by hand.
    # 兜底：如果服务端没有拆分字段、思考过程仍嵌在 content 里，就手动剥出来。
    if not reasoning and content:
        m = re.search(r"<think>(.*?)</think>", content, re.S)
        if m:
            reasoning = m.group(1)
        m2 = re.search(r"<\|im_start\|>think\n(.*?)<\|im_end\|>", content, re.S)
        if m2:
            reasoning = m2.group(1)

    # Append prompt + reasoning to thinking.log. This is the only window into why the
    # model produced a wrong grid, and a logging failure must never kill the run.
    # 把 prompt 和推理过程追加到 thinking.log。这是唯一能看出模型为什么给出错误网格的
    # 窗口；日志失败绝不能中断整个运行。
    try:
        log_path = "/kaggle/working/thinking.log" if os.path.isdir("/kaggle/working") \
                   else os.path.join(os.getcwd(), "thinking.log")
        with open(log_path, "a", encoding="utf-8") as f:
            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            f.write(f"[{ts}] Prompt: {prompt[:200]}\n")
            f.write(f"Reasoning:\n{reasoning}\n")
            f.write("-" * 60 + "\n")
        print("log ->", log_path)
    except Exception as e:
        print("log fail:", e)

    return content


# ===========================================================================
# 2) Configuration, and the overall deadline
# 2) 配置，以及整体时限
# ===========================================================================

# CHALLENGES_FILE = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
# CHALLENGES_FILE = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_training_challenges.json"
CHALLENGES_FILE = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
# SOLUTIONS_FILE  = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json"
SOLUTIONS_FILE = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_training_solutions.json"
OUTPUT_DIR      = "/kaggle/working/output"
SUBMISSION_FILE = "/kaggle/working/submission.json"
SCORE_FILE      = "/kaggle/working/score.json"

# MODE: "eval" solves, aggregates and scores; "submit" stops after aggregating.
# N_PARALLEL must not exceed --max-num-seqs from the server cell.
# MAX_RETRIES is 0 here: each task is attempted exactly once. Raise it to 2 for three
# attempts, but note that temperature=0.0 makes a retry reproduce the same answer.
# SKIP_EXISTING makes a re-run resume: tasks with a result file are skipped.
# MODE："eval" 求解 + 汇总 + 打分；"submit" 只到汇总为止。
# N_PARALLEL 不能超过服务 cell 里的 --max-num-seqs。
# 这里的 MAX_RETRIES 是 0：每个任务只尝试一次。设成 2 就是三次尝试，
# 但注意 temperature=0.0 会让重试复现同样的答案。
# SKIP_EXISTING 让重跑变成断点续跑：已有结果文件的任务会跳过。
MODE          = "eval"
N_PARALLEL    = 16
SKIP_EXISTING = True
MAX_RETRIES   = 0
RETRY_SLEEP   = 3.0
PROGRESS_REFRESH = 5.0

# Overall deadline, measured from the first cell of the notebook. Once it passes, no
# new task is started, requests already in flight are cut off by their timeout, and
# whatever is unfinished is filled in by the aggregation fallback — so the submission
# is still complete.
# A notebook can set RUN_START_TS / DEADLINE_MIN before this cell to override them.
# 整体时限，从 notebook 的第一个 cell 开始计时。一旦到点，就不再开新任务，
# 在飞的请求会被各自的超时掐断，没跑完的由汇总阶段的兜底补齐 ——
# 所以提交文件依然是完整的。
# notebook 可以在这个 cell 之前设置 RUN_START_TS / DEADLINE_MIN 来覆盖默认值。
RUN_START_TS  = globals().get("RUN_START_TS") or time.time()
# Overall deadline in minutes from RUN_START_TS. 690 min = 11h30m.
# 整体时限，从 RUN_START_TS 起算的分钟数。690 分钟 = 11 小时 30 分。
DEADLINE_MIN  = globals().get("DEADLINE_MIN") or 690


def deadline_ts():
    # Absolute wall-clock timestamp of the deadline.
    # 截止时刻的绝对时间戳。
    return RUN_START_TS + DEADLINE_MIN * 60


def seconds_left():
    # Seconds remaining before the deadline; negative once it has passed.
    # 距截止还剩多少秒；到点后为负。
    return deadline_ts() - time.time()


def _fmt_left():
    s = max(0, int(seconds_left()))
    return f"{s // 3600}h{s % 3600 // 60}m"


def _chat_timeout(default=21600):
    # Per-request timeout, clamped to the remaining overall budget so a late request
    # cannot hang on a socket past the deadline. chat() calls this before it is
    # defined in source order — fine, Python resolves names at call time.
    # 单次请求超时，压到整体剩余时限之内，免得临近截止还卡在 socket 上。
    # chat() 在源码顺序上先于它定义就调用了 —— 没关系，Python 在调用时才解析名字。
    return max(5, int(min(default, seconds_left())))


# ===========================================================================
# 3) JSON and grid helpers
# 3) JSON 与网格工具
# ===========================================================================

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)


def _fmt_sec(seconds):
    # Render a duration as "3m25.6s" for progress and summary lines.
    # 把秒数格式化成 "3m25.6s"，用于进度和汇总行。
    m = int(seconds) // 60
    s = seconds - m * 60
    return f"{m}m{s:.1f}s" if m else f"{seconds:.1f}s"


def _normalize_grid(g):
    # Coerce anything that might be a grid into a rectangular 2-D list of ints, and
    # return None if it is not one. Rejecting malformed grids here matters: Kaggle's
    # scorer rejects an entire malformed submission rather than scoring it partially.
    # Tolerates JSON strings, floats like 1.0, and dicts wrapping the grid.
    # 把任何可能是网格的东西规整成矩形的二维 int 列表，不符合就返回 None。
    # 在这里拒绝非法网格很重要：Kaggle 的打分器会直接拒收整份非法提交，
    # 而不是只扣那一部分分。兼容 JSON 字符串、1.0 这类浮点数、以及包着网格的 dict。
    if g is None:
        return None
    if isinstance(g, str):
        s = g.strip().strip("'\"")
        try:
            g = json.loads(s)
        except Exception:
            return None
    if isinstance(g, dict):
        for k in ("grid", "output", "attempt_1", "solution"):
            if k in g:
                return _normalize_grid(g[k])
        return None
    if not isinstance(g, list):
        return None
    rows = []
    for r in g:
        if isinstance(r, str):
            try:
                r = json.loads(r.strip().strip("'\""))
            except Exception:
                return None
        if not isinstance(r, list):
            return None
        row = []
        for v in r:
            if isinstance(v, float) and v.is_integer():
                v = int(v)
            if not isinstance(v, int) or isinstance(v, bool):
                return None
            row.append(v)
        rows.append(row)
    if not rows:
        return None
    width = len(rows[0])
    if width == 0 or any(len(row) != width for row in rows):
        return None
    return rows


def _expected_attempt_keys(n):
    # How many attempt_* keys a task with n outputs must supply: 1 output -> 2 keys
    # (two candidates for that single output), n >= 2 -> n keys (one per output).
    # 一个有 n 个输出的任务必须提供几个 attempt_* 键：1 个输出 -> 2 个键
    # （同一个输出的两个候选），n >= 2 -> n 个键（每个输出一个）。
    return max(2, n)


# ===========================================================================
# 4) Prompt construction
# 4) Prompt 构造
# ===========================================================================

def convert_payload_to_text(payload_dict):
    # Render a task as the flat line format the model was tuned on: a literal marker
    # (train / input / output / test) followed by each grid row as a run of digits,
    # e.g. "0123" rather than [0,1,2,3]. Much cheaper in tokens than JSON.
    # The marker words are part of the wire format — do not rename them.
    #
    # Unlike a single-input serialization, every test input is emitted as its own
    # "input" block, so a task with three test inputs produces three of them in order.
    # 把任务渲染成模型训练时使用的扁平行格式：先是字面标记
    # （train / input / output / test），然后每个网格行是一串数字，
    # 例如 "0123" 而不是 [0,1,2,3]。这比 JSON 省很多 token。
    # 那些标记词是协议的一部分 —— 不要改名。
    #
    # 与只序列化单个输入的写法不同，这里每个 test 输入都会输出一个独立的 "input" 块，
    # 所以有三个 test 输入的任务会按顺序产出三段。
    lines = []
    for task_id, data in payload_dict.items():
        lines.append(task_id)
        lines.append("train")
        for ex in data["train"]:
            lines.append("input")
            for row in ex["input"]:
                lines.append(''.join(map(str, row)))
            lines.append("output")
            for row in ex["output"]:
                lines.append(''.join(map(str, row)))
        lines.append("test")
        for te in data["test"]:
            lines.append("input")
            for row in te["input"]:
                lines.append(''.join(map(str, row)))
        lines.append("")
    return '\n'.join(lines).rstrip('\n')


def build_prompt(task_id, task):
    # Produces: [task as text] + [instructions].
    # The instructions below are intentionally in Chinese. The reported scores were
    # produced with exactly this prompt, and translating it changes model behaviour —
    # re-run the eval before trusting the number if you do translate it.
    # 产出：[任务文本] + [指令]。
    # 下面的指令刻意使用中文。报告出来的分数就是用这份 prompt 跑出来的，
    # 翻译它会改变模型行为 —— 如果你翻译了，请先重跑评测再相信那个数字。
    payload_task = {
        "train": task["train"],
        "test": [{"input": ex["input"]} for ex in task["test"]],
    }
    payload = json.dumps({task_id: payload_task}, ensure_ascii=False, separators=(",", ":"))

    payload_dict = json.loads(payload)
    text_output = convert_payload_to_text(payload_dict)

    n = len(task["test"])
    sample = [[0, 0], [0, 0]]

    if n == 1:
        example = json.dumps({task_id: {"attempt_1": sample, "attempt_2": sample}},
                             ensure_ascii=False, separators=(",", ":"))
        body = (
            "这个任务只需要预测 1 个输出。请直接输出一个json，不要输出其他任何文字，格式如下：\n"
            f"{example}\n"
            "含义：顶层键是任务id，值是包含两个键的对象：\n"
            "  attempt_1 —— 你最有把握的答案（预测的二维网格）；\n"
            "  attempt_2 —— 给同一个输出的第二个候选答案（可以与 attempt_1 相同，也可以不同）。\n"
            "不管想不想得出来，attempt_1 和 attempt_2 这两个键都必须给出。"
        )
    else:
        ex_obj = {task_id: {f"attempt_{i}": sample for i in range(1, n + 1)}}
        example = json.dumps(ex_obj, ensure_ascii=False, separators=(",", ":"))
        body = (
            f"这个任务需要预测 {n} 个输出（test 里有 {n} 个 input，一一对应）。"
            "请直接输出一个json，不要输出其他任何文字，格式如下：\n"
            f"{example}\n"
            "含义：顶层键是任务id，值是一个对象，必须恰好给出 "
            f"attempt_1 到 attempt_{n} 这 {n} 个键：\n"
            "  attempt_1 对应 test 中第 1 个 input 的输出；\n"
            "  attempt_2 对应第 2 个 input 的输出；依此类推，顺序不能错乱。\n"
            "打分时 attempt_i 只会与第 i 个输出比对，错位不记分。"
        )

    # The tail instruction asks for "|" separators on wide rows. Without it the model
    # miscounts columns on grids wider than 8 cells and emits ragged rows, which
    # _normalize_grid then rejects — a pure formatting loss, not a reasoning failure.
    # 结尾的指令要求宽网格用 "|" 分隔。没有它的话，模型在宽度超过 8 格的网格上会数错列数、
    # 输出参差不齐的行，然后被 _normalize_grid 拒掉 —— 那是纯格式损失，不是推理失败。
    tail = (
        "网格是二维整数数组，元素是0~9的颜色值（示例里的0只是占位示意，不代表真实答案）。当你输出的网格的每行的元素大于8时，每4个元素用|符号分隔，以免数错数字的个数，最终的网格一定是一个每行元素个数都相等每列元素个数都相等的网格，请你仔细确认后再输出"
    )
    return text_output + "\n\n" + body + "\n" + tail


# ===========================================================================
# 5) Response normalization
# 5) 响应归一化
# ===========================================================================

def _slot_set(slots, idx, g, filled):
    gg = _normalize_grid(g)
    if idx is not None and 0 <= idx < len(slots) and gg is not None:
        slots[idx] = gg
        filled[0] = True


def _parse_attempt_index(key):
    # "attempt_3" -> 2. Returns None for anything that is not an attempt key.
    # "attempt_3" -> 2。不是 attempt 键的一律返回 None。
    if isinstance(key, str):
        m = re.fullmatch(r"attempt_(\d+)", key)
        if m:
            return int(m.group(1)) - 1
    return None


def _extract_from_text(text):
    # Dig a JSON object out of the model's raw text. In practice it comes back wrapped
    # in prose, inside ```json fences, or as a Python single-quoted literal — so try
    # strict JSON first and fall back to ast.literal_eval.
    # 从模型的原始文本里挖出 JSON 对象。实践中它会带前后文、被 ```json 围栏包住、
    # 或者写成 Python 单引号字面量 —— 所以先试严格 JSON，再回退到 ast.literal_eval。
    if not isinstance(text, str):
        return text
    text = text.strip().lstrip("﻿")
    if not text:
        return None
    candidates = [text]
    for m in re.finditer(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.S):
        candidates.append(m.group(1))
    i, j = text.find("{"), text.rfind("}")
    if i != -1 and j != -1 and j > i:
        candidates.append(text[i:j + 1])
    for c in candidates:
        s = c.strip().strip("'\"")
        try:
            return json.loads(s)
        except Exception:
            pass
        try:
            import ast
            return ast.literal_eval(s)
        except Exception:
            continue
    return None


def _container_to_slots(container, k):
    # Map whatever container shape came back onto a fixed list of k slots. Handles a
    # dict of attempt keys, a bare list of grids, and a single-element list wrapping
    # the dict. Returns None when no valid grid was found at all.
    # 把无论什么形状的容器映射到固定的 k 个槽位。兼容 attempt 键组成的 dict、
    # 裸的网格列表、以及只含一个 dict 的列表。一个有效网格都没有时返回 None。
    slots = [None] * k
    filled = [False]

    if isinstance(container, dict):
        got = False
        for key, val in container.items():
            idx = _parse_attempt_index(key)
            if idx is not None:
                _slot_set(slots, idx, val, filled)
                got = True
        if not got:
            for key in ("output", "prediction", "answer", "grid"):
                if key in container:
                    _slot_set(slots, 0, container[key], filled)
                    break
    elif isinstance(container, list) and container:
        if isinstance(container[0], dict):
            if len(container) == 1:
                return _container_to_slots(container[0], k)
            for i, e in enumerate(container):
                if i >= k:
                    break
                if isinstance(e, dict) and "attempt_1" in e:
                    _slot_set(slots, i, e["attempt_1"], filled)
                else:
                    _slot_set(slots, i, e, filled)
        else:
            for i, g in enumerate(container):
                if i >= k:
                    break
                _slot_set(slots, i, g, filled)
    return slots if filled[0] else None


def normalize_response(task_id, task, resp):
    # The contract every downstream step relies on: return exactly max(2, n_outputs)
    # slots, each either a valid grid or None. Shape:
    #   {task_id: {"attempt_1": grid|None, "attempt_2": grid|None, ...}}
    # Returns None if nothing usable could be parsed. Note that a partial answer is
    # kept: slots the model did not fill stay None rather than failing the whole task.
    # 下游每一步都依赖这个契约：恰好返回 max(2, 输出个数) 个槽位，
    # 每个要么是合法网格，要么是 None。形状为：
    #   {task_id: {"attempt_1": grid|None, "attempt_2": grid|None, ...}}
    # 完全解析不出可用内容时返回 None。注意部分答案会被保留：
    # 模型没填出来的槽位保持 None，而不是让整个任务失败。
    n = len(task["test"])
    k = _expected_attempt_keys(n)

    if isinstance(resp, str):
        resp = _extract_from_text(resp)
        if resp is None:
            return None

    container = None
    if isinstance(resp, dict):
        if task_id in resp:
            container = resp[task_id]
        elif any(_parse_attempt_index(key) is not None for key in resp):
            container = resp
        else:
            cands = [v for v in resp.values() if isinstance(v, (dict, list))]
            container = cands[0] if len(cands) == 1 else None
    else:
        container = resp

    slots = _container_to_slots(container, k)
    if slots is None:
        return None
    return {task_id: {f"attempt_{i + 1}": slots[i] for i in range(k)}}


# ===========================================================================
# 6) Per-task logging and failure diagnosis
# 6) 逐任务日志与失败诊断
# ===========================================================================

def _append_log(out_dir, task_id, msg):
    # One timestamped line per event into output/solve.log. Swallows its own errors.
    # 每个事件往 output/solve.log 追加一行带时间戳的记录。自身的错误一律吞掉。
    try:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        ts = time.strftime("%Y-%m-%d %H:%M:%S")
        with open(out_dir / "solve.log", "a", encoding="utf-8") as f:
            f.write(f"[{ts}] task={task_id}  {msg}\n")
    except Exception:
        pass


def _save_raw(out_dir, task_id, raw):
    # Keep the raw model output whenever parsing fails, so a human can see why.
    # 解析失败时保留模型原始输出，方便人工查看原因。
    try:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        with open(out_dir / f"raw_{task_id}.txt", "w", encoding="utf-8") as f:
            f.write(raw if isinstance(raw, str) else json.dumps(raw, ensure_ascii=False))
    except Exception:
        pass


def _classify_error(e):
    # Turn an exception from chat() into a plain-English cause, separating transport
    # failures (server problems) from everything else.
    # 把 chat() 抛出的异常翻译成人话原因，把传输层故障（服务端问题）和其他问题分开。
    import urllib.error as _ue
    if isinstance(e, TimeoutError):
        return "timeout: the server did not return in time (hung, or generation was interrupted)"
    if isinstance(e, ConnectionRefusedError):
        return "connection refused: nothing is listening on 127.0.0.1:8000"
    if isinstance(e, ConnectionError):
        return f"connection dropped ({type(e).__name__}): the server may have been killed or crashed"
    if isinstance(e, _ue.HTTPError):
        return f"HTTP {e.code} {e.reason}: the server returned an error (overloaded, or the process died)"
    if isinstance(e, _ue.URLError):
        return f"URLError: {e.reason}"
    return f"{type(e).__name__}: {e}"


def _why_no_grid(task_id, task, resp):
    # Explain which of the three failure classes this was: no JSON at all (usually a
    # max_tokens truncation while still thinking), JSON present but with wrong attempt
    # keys, or right keys but invalid grid shapes. Prints the offending keys and types.
    # 说明这属于三类失败中的哪一种：完全没有 JSON（通常是还在思考就耗尽了
    # max_tokens）、有 JSON 但 attempt 键不对、或者键对了但网格形状非法。
    # 会打印出有问题的键及其类型。
    if resp is None:
        return "chat returned None"
    if isinstance(resp, str):
        s = resp.strip()
        if not s:
            return "chat returned empty content (generation was probably interrupted by vLLM)"
        has_think = ("<think" in s) or ("|im_start|>think" in s)
        obj = _extract_from_text(s)
        if obj is None:
            hint = " the text looks like thinking only, with no final JSON" if has_think else ""
            return f"no JSON object could be extracted from the text (truncated by max_tokens, or malformed){hint}"
        resp = obj

    detail = []
    n = len(task["test"])
    k = _expected_attempt_keys(n)
    container = None
    if isinstance(resp, dict):
        if task_id in resp:
            container = resp[task_id]
        elif any(_parse_attempt_index(key) is not None for key in resp):
            container = resp
    else:
        container = resp
    tmp = container
    if isinstance(container, list) and len(container) == 1 and isinstance(container[0], dict):
        tmp = container[0]
    if isinstance(tmp, dict):
        keys = [kk for kk in tmp if _parse_attempt_index(kk) is not None] or list(tmp)[:8]
        detail.append("keys: " + ", ".join(str(x) for x in keys)[:120])
        for kk in keys[:k + 1]:
            v = tmp.get(kk)
            if isinstance(v, list):
                detail.append(f"{kk}: length={len(v)} (expected rows of columns)"
                              if v and isinstance(v[0], list) else f"{kk}: {v}")
            elif isinstance(v, dict):
                detail.append(f"{kk}: is an object, keys={list(v)[:6]}")
            else:
                detail.append(f"{kk}: type {type(v).__name__}")
    return "found JSON but no valid grid (" + "; ".join(detail)[:300] + "), see the raw file"


# ===========================================================================
# 7) Parallel solving, bounded by the deadline
# 7) 并行求解，受整体时限约束
# ===========================================================================

def _solve_one(task_id, task, out_dir):
    # Solve a single task: build the prompt, call chat, normalize, write the result.
    # On total failure it deliberately writes NO result file, which is what makes
    # SKIP_EXISTING a correct resume mechanism — a missing file is retried next run.
    # 求解单个任务：构造 prompt、调用 chat、归一化、写出结果。
    # 彻底失败时故意不写结果文件，这正是 SKIP_EXISTING 能作为正确的断点续跑机制的
    # 原因 —— 缺文件的任务下次会自动重试。
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"result_{task_id}.json"
    prompt = build_prompt(task_id, task)

    _append_log(out_dir, task_id, f"start (prompt {len(prompt)} chars)")
    last_err = None
    for attempt in range(MAX_RETRIES + 1):
        # Past the deadline, stop opening new requests.
        # 过了截止时刻就不再开新请求。
        if seconds_left() <= 0:
            last_err = f"deadline reached ({DEADLINE_MIN} min), giving up"
            break
        tag = f"try {attempt + 1}/{MAX_RETRIES + 1}"
        try:
            resp = chat(prompt)
        except Exception as e:
            last_err = "chat raised: " + _classify_error(e)
            _append_log(out_dir, task_id, f"{tag} {last_err}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_SLEEP)
            continue

        raw = resp if isinstance(resp, str) else json.dumps(resp, ensure_ascii=False)
        _append_log(out_dir, task_id, f"{tag} chat returned ({len(raw)} chars)")

        payload = None
        try:
            payload = normalize_response(task_id, task, resp)
        except Exception as e:
            last_err = "normalization raised: " + _classify_error(e)

        if payload is None:
            if last_err is None:
                last_err = _why_no_grid(task_id, task, resp)
            _append_log(out_dir, task_id, f"{tag} no usable answer: {last_err}")
            _save_raw(out_dir, task_id, raw)
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_SLEEP)
            continue

        save_json(payload, out_path)
        _append_log(out_dir, task_id, f"{tag} ok -> wrote result_{task_id}.json")
        # Hand the payload back so the caller can write it straight into the live
        # submission without re-reading the file from disk.
        # 把 payload 回传给调用方，让它直接写进实时提交文件，不必再读一次磁盘。
        return "ok", task_id, payload

    _append_log(out_dir, task_id, f"gave up, no result file written: {last_err}")
    return "fail", task_id, last_err


def solve_challenges(challenges=None, n_parallel=None, out_dir=None, skip_existing=None,
                     submission=None, submission_file=None):
    # Solve every task with a rolling pool of n_parallel workers.
    #
    # Tasks are sorted by prompt length, shortest first, and submitted to a
    # ThreadPoolExecutor. Its work queue is FIFO, so short prompts start first, and
    # because the pool is rolling rather than batched, no slot ever idles waiting for
    # the slowest member of a batch. Generation length varies by more than an order of
    # magnitude across ARC tasks, so this matters a lot for wall-clock time.
    #
    # Working against that is the deadline: once seconds_left() hits zero, queued tasks
    # are abandoned as "deadline" instead of being started, and aggregation fills them.
    #
    # 用 n_parallel 个 worker 的滚动池求解全部任务。
    #
    # 任务按 prompt 长度从短到长排序后提交给 ThreadPoolExecutor。它的工作队列是
    # FIFO，所以短 prompt 先开始；又因为这是滚动池而非分批，没有任何槽位会为了等
    # 一批里最慢的那个而空转。ARC 任务的生成长度相差一个数量级以上，
    # 这对墙钟时间影响很大。
    #
    # 与之制衡的是整体时限：一旦 seconds_left() 归零，
    # 排队中的任务会被标记为 "deadline" 直接放弃，而不是被启动，之后由汇总补齐。
    #
    # When `submission` is supplied it is updated in place and re-saved on every
    # success, so submission.json on disk stays valid and keeps pace with the run — a
    # run cut short by the deadline still leaves a complete, submittable file behind.
    # 传入 `submission` 时，每有一个任务成功就地更新并重新保存一次，
    # 于是磁盘上的 submission.json 始终合法、且始终跟得上进度 ——
    # 即使因为到点而被中断，留下的也仍是一份完整可提交的文件。
    #
    # Returns {"ok": count, "skip": count, "fail": [...], "deadline": [...]}.
    # 返回 {"ok": 成功数, "skip": 跳过数, "fail": [...], "deadline": [...]}。
    if challenges is None:
        challenges = load_json(CHALLENGES_FILE)
    if n_parallel is None:
        n_parallel = N_PARALLEL
    if out_dir is None:
        out_dir = OUTPUT_DIR
    if skip_existing is None:
        skip_existing = SKIP_EXISTING
    if submission_file is None:
        submission_file = SUBMISSION_FILE
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    t_start = time.perf_counter()

    # Pre-compute prompt lengths for the sort. build_prompt is pure string assembly,
    # so this dry run is cheap. Tasks whose prompt cannot be built sort last.
    # 预先算出各 prompt 长度用于排序。build_prompt 只是字符串拼接，
    # 这个预跑很便宜。连 prompt 都构造不出来的任务排在最后。
    len_of = {}
    for tid in challenges:
        try:
            len_of[tid] = len(build_prompt(tid, challenges[tid]))
        except Exception:
            len_of[tid] = None
    ids = sorted(challenges, key=lambda t: (len_of[t] is None, len_of[t] or 0))

    total = len(ids)
    known = [v for v in len_of.values() if v is not None]
    print(f"[solve] tasks: {total} | concurrency {n_parallel} (rolling refill) | output dir: {out_dir}")
    print(f"[solve] deadline {DEADLINE_MIN} min, {_fmt_left()} left "
          f"(no new tasks after that; the rest is filled in at aggregation)")
    if known:
        print(f"[solve] order: prompt length ascending ({min(known)} ~ {max(known)} chars)")
    if skip_existing:
        already = sum((Path(out_dir) / f"result_{tid}.json").exists() for tid in ids)
        print(f"[solve] existing results that will be skipped: {already}")

    lock = threading.Lock()
    stats = {"ok": 0, "skip": 0, "fail": [], "deadline": []}
    tick = {"done": 0}
    running = set()

    # Background thread printing progress every PROGRESS_REFRESH seconds, with a
    # running mean and an ETA.
    # 后台线程，每隔 PROGRESS_REFRESH 秒打印一次进度，带滑动均值和预计剩余时间。
    def _monitor():
        while True:
            if stop_event.wait(PROGRESS_REFRESH):
                return
            elapsed = time.perf_counter() - t_start
            with lock:
                run_now = sorted(running)
                done_now = tick["done"]
                ok_now = stats["ok"]
                fail_now = len(stats["fail"])
                skip_now = stats["skip"]
            if done_now > 0:
                avg = elapsed / done_now
                eta = _fmt_sec(avg * (total - done_now))
                avg_s = f"{avg:.1f}s/task"
            else:
                avg, eta, avg_s = None, "?", ""
            print(f"[progress] elapsed {_fmt_sec(elapsed)} | done {done_now}/{total} "
                  f"| ok {ok_now} | fail {fail_now} | skip {skip_now} "
                  f"| in flight ({len(run_now)})"
                  + (f" | avg {avg_s} | ETA {eta}" if avg else ""), flush=True)

    def _work(tid):
        out_path = Path(out_dir) / f"result_{tid}.json"
        if skip_existing and out_path.exists():
            with lock:
                stats["skip"] += 1
                tick["done"] += 1
            return "skip", tid
        # Past the deadline: do not start this task at all.
        # 过了截止时刻：这个任务根本不启动。
        if seconds_left() <= 0:
            with lock:
                stats["deadline"].append(tid)
                tick["done"] += 1
            return "deadline", tid
        t0 = time.perf_counter()
        with lock:
            running.add(tid)
        try:
            result = _solve_one(tid, challenges[tid], out_dir)
        finally:
            with lock:
                running.discard(tid)
        dt = time.perf_counter() - t0
        status = result[0]
        with lock:
            tick["done"] += 1
            if status == "ok":
                stats["ok"] += 1
                # Swap this task's fallback slots for the real answer and persist, so
                # the file on disk is never stale and never missing a slot.
                # 把这个任务的兜底槽位换成真实答案并落盘，
                # 保证磁盘上的文件既不过期、也不缺槽位。
                if submission is not None:
                    submission[tid] = _merge_answer(_fallback_entry(challenges[tid]),
                                                    result[2][tid])
                    save_json(submission, submission_file)
            elif status == "deadline":
                stats["deadline"].append(tid)
            else:
                stats["fail"].append((tid, result[2]))
            msg = (f"[solve] {tick['done']}/{total}  task={tid}  -> {status}"
                   f"  {dt:.1f}s"
                   + (f"  error: {result[2]}" if status == "fail" else ""))
            print(msg, flush=True)
        return status, tid

    stop_event = threading.Event()
    monitor = threading.Thread(target=_monitor, daemon=True)
    if PROGRESS_REFRESH and PROGRESS_REFRESH > 0:
        monitor.start()

    # The pool stays at n_parallel requests in flight until the queue drains.
    # 队列排空前，池子里始终保持 n_parallel 个请求在飞。
    with ThreadPoolExecutor(max_workers=n_parallel) as ex:
        futures = [ex.submit(_work, tid) for tid in ids]
        for fut in as_completed(futures):
            fut.result()

    if PROGRESS_REFRESH and PROGRESS_REFRESH > 0:
        stop_event.set()
        monitor.join(timeout=1)

    total_t = time.perf_counter() - t_start
    print(f"[solve] done: ok {stats['ok']} | skip {stats['skip']} "
          f"| deadline {len(stats['deadline'])} | fail {len(stats['fail'])}"
          f" | total {_fmt_sec(total_t)}")
    if stats["deadline"]:
        print(f"[solve] abandoned at the {DEADLINE_MIN} min deadline, filled in at "
              f"aggregation:", stats["deadline"][:20])
    if stats["fail"]:
        print("[solve] failed tasks (no result file written; a rerun retries them):",
              [t for t, _ in stats["fail"]][:20])
    return stats


# ===========================================================================
# 8) Aggregation, and the rule-based fallback
# 8) 汇总，以及基于规则的兜底
# ===========================================================================

def _fallback_grids(task, k):
    # A guess for a task that was never solved, using only patterns visible in its own
    # train pairs. No model call, and the result is always a structurally valid grid:
    #   1. every train output is identical  -> a constant task, reuse that grid
    #   2. some train output equals its own input -> identity-like, reuse the test input
    #   3. otherwise -> the most common train output
    # For a 1-output task the second slot is given a different candidate, buying the
    # free second guess that scoring allows there.
    # 为一个没解出来的任务猜答案，只用它自己 train 里可见的规律，不调用模型，
    # 且结果一定是结构合法的网格：
    #   1. train 的输出全都一样 -> 常数任务，直接用那个网格
    #   2. 有 train 的输出等于自己的输入 -> 类恒等任务，用对应的 test 输入
    #   3. 其余 -> 用出现次数最多的那个 train 输出
    # 对 1 个输出的任务，第二个槽位会给一个不同的候选，
    # 用上打分规则在这里允许的那次免费重猜。
    ins = [ex["input"] for ex in task["train"]]
    outs = [ex["output"] for ex in task["train"]]
    test = [ex["input"] for ex in task["test"]]
    if not test:
        return [None] * k
    constant = outs[0] if outs and all(o == outs[0] for o in outs) else None
    common = (json.loads(collections.Counter(json.dumps(o) for o in outs).most_common(1)[0][0])
              if outs else None)
    identity = any(o == a for o, a in zip(outs, ins))

    def guess(i):
        t = test[i] if i < len(test) else test[-1]
        if identity:
            return t
        return constant if constant is not None else (common if common is not None else t)

    grids = [guess(i) for i in range(k)]
    if k == 2 and grids[1] == grids[0]:
        alt = test[0] if grids[0] != test[0] else common
        if alt is not None and alt != grids[0]:
            grids[1] = alt
    return grids

def _placeholder_grid():
    # Last-resort filler: a valid but certainly-wrong all-zero grid. Only used when
    # even the rule-based fallback cannot produce a usable grid for a slot.
    # 最后兜底用的填充值：一个合法但必然错误的全 0 网格。
    # 只有当连基于规则的兜底都产不出可用网格时才会用到它。
    return [[0, 0], [0, 0]]


def _fallback_entry(task):
    # A complete, structurally valid entry for one task, built by _fallback_grids.
    # Any slot that still comes back invalid degrades to a zeros grid, so this never
    # returns None and never returns a malformed grid.
    # 为一个任务构造完整且结构合法的条目，由 _fallback_grids 生成。
    # 任何仍然不合法的槽位退化成全 0 网格，所以这里绝不会返回 None 或畸形网格。
    k = _expected_attempt_keys(len(task["test"]))
    entry = {}
    for i, g in enumerate(_fallback_grids(task, k)):
        grid = _normalize_grid(g)
        entry[f"attempt_{i + 1}"] = grid if grid is not None else _placeholder_grid()
    for i in range(len(entry), k):
        entry[f"attempt_{i + 1}"] = _placeholder_grid()
    return entry


def _merge_answer(entry, answer):
    # Overwrite a fallback entry with the valid grids from one normalized answer.
    # Slots the model did not fill keep their fallback value instead of becoming null.
    # 用一份归一化答案里的合法网格覆盖兜底条目。
    # 模型没填出来的槽位保留兜底值，而不是变成 null。
    for key, val in answer.items():
        if key in entry:
            grid = _normalize_grid(val)
            if grid is not None:
                entry[key] = grid
    return entry



def aggregate_to_submission(challenges=None, out_dir=None, submission_file=None, fallback=True):
    # Collect output/result_*.json into a single submission.json.
    #
    # Tasks with no result file — never run, failed, or abandoned at the deadline — are
    # filled by _fallback_grids, so the file written here is always complete: every
    # task, every attempt slot, a valid grid, never null. That is what makes the
    # notebook safe to submit even if it is cut short.
    #
    # It reads files only, which is why it is free to re-run after a scoring or
    # normalization fix, with no model calls.
    #
    # 把 output/result_*.json 收拢成单个 submission.json。
    #
    # 没有结果文件的任务 —— 没跑、失败、或到点被放弃的 —— 都由 _fallback_grids 补齐，
    # 所以这里写出的文件永远是完整的：每个任务、每个 attempt 槽位都是合法网格，
    # 绝不是 null。正因如此，即使 notebook 被提前中断，提交也是安全的。
    #
    # 它只读文件，所以修了打分或归一化之后可以随便重跑，不需要任何模型调用。
    if challenges is None:
        challenges = load_json(CHALLENGES_FILE)
    if out_dir is None:
        out_dir = OUTPUT_DIR
    if submission_file is None:
        submission_file = SUBMISSION_FILE
    out_dir = Path(out_dir)

    sub = {}
    missing = []
    for tid, task in challenges.items():
        k = _expected_attempt_keys(len(task["test"]))
        path = out_dir / f"result_{tid}.json"
        # Always start from a complete, valid entry, then overwrite the slots that hold
        # a real answer. A task whose result file is missing, unreadable, or only
        # partially valid therefore degrades to its fallback grid instead of to a null
        # — and a null is what would make the whole submission unscoreable.
        # 永远从一个完整合法的条目开始，再覆盖确实有答案的槽位。
        # 于是结果文件缺失、读不出、或只有部分合法的任务，会退化成它的兜底网格而不是
        # null —— 而 null 正是会让整份提交无法计分的东西。
        entry = _fallback_entry(task) if fallback else {
            f"attempt_{i + 1}": _placeholder_grid() for i in range(k)}
        if path.exists():
            # A result file can be truncated if the process was killed mid-write, so a
            # corrupt file has to degrade to the fallback rather than crash aggregation.
            # 进程被中途杀掉时，结果文件可能只写了一半 ——
            # 所以损坏的文件必须退化成兜底，而不是让整个汇总崩掉。
            try:
                data = load_json(path)
            except Exception:
                data = {}
            val = data.get(tid, {})
            if isinstance(val, dict):
                _merge_answer(entry, val)
        else:
            missing.append(tid)
        sub[tid] = entry

    # Final safety net: repair anything still not a valid grid.
    # 最后的兜底：修复任何仍然不是合法网格的槽位。
    repaired = ensure_submission_complete(sub, challenges)
    save_json(sub, submission_file)

    print(f"[aggregate] {len(sub)} tasks -> {submission_file}")
    if missing:
        print(f"[aggregate] {len(missing)} tasks have no result file, "
              f"filled by the fallback: {missing[:20]}")
    empty = sum(1 for v in sub.values() for g in v.values() if g is None)
    print(f"[aggregate] completeness: {len(sub)} tasks / "
          f"{sum(len(v) for v in sub.values())} attempt slots, null {empty}"
          + (f", repaired {repaired}" if repaired else ""))
    return sub


def ensure_submission_complete(submission, challenges):
    # Last line of defence, run after solving finishes and after the deadline. Walks
    # the whole submission and replaces anything that is not a valid grid with a
    # fallback grid, so a missing or malformed answer can never render the file
    # unscoreable. Returns how many slots it had to repair — 0 is the healthy value.
    # 最后一道兜底，在求解结束、以及截止时间之后运行。遍历整份提交，
    # 把任何不是合法网格的槽位换成兜底网格，
    # 保证缺失或格式错误的答案绝不会让整份文件无法计分。
    # 返回需要修复的槽位数 —— 0 才是健康值。
    repaired = 0
    for tid, task in challenges.items():
        k = _expected_attempt_keys(len(task["test"]))
        entry = submission.get(tid)
        if not isinstance(entry, dict):
            submission[tid] = _fallback_entry(task)
            repaired += k
            continue
        bad = [f"attempt_{i + 1}" for i in range(k)
               if _normalize_grid(entry.get(f"attempt_{i + 1}")) is None]
        if not bad:
            continue
        fb = _fallback_entry(task)          # computed once, only when needed
        for key in bad:
            entry[key] = fb[key]
            repaired += 1
    return repaired


# ===========================================================================
# 9) Scoring
# 9) 打分
# ===========================================================================

def _get_attempt(pred, idx):
    # Pull the idx-th (0-based) attempt out of one submission entry, accepting both
    # the current dict shape and the legacy list-of-containers shape.
    # 从一条提交记录里取出第 idx 个（0 起）attempt，
    # 同时兼容当前的 dict 形状和旧的"容器列表"形状。
    if isinstance(pred, dict):
        return pred.get(f"attempt_{idx + 1}")
    if isinstance(pred, list) and pred:
        if len(pred) == 1 and isinstance(pred[0], dict):
            return pred[0].get(f"attempt_{idx + 1}")
        if idx < len(pred):
            e = pred[idx]
            return e.get("attempt_1") if isinstance(e, dict) else e
    return None


def score_submission(submission=None, solutions=None):
    # 100 points split evenly across every output that must be predicted (not across
    # tasks), so a 3-output task is worth three times a 1-output task.
    #   - 1 output: attempt_1 OR attempt_2 matching scores it (two candidates).
    #   - n >= 2 outputs: only attempt_i matching output i scores; off-by-one scores 0.
    # Returns (score, correct, total_required, per_output, task_detail).
    # 满分 100 均分到每一个需要预测的输出上（不是按任务分），
    # 所以一个 3 输出的任务权重是 1 输出任务的 3 倍。
    #   - 1 个输出：attempt_1 或 attempt_2 命中即得分（两个候选）。
    #   - n >= 2 个输出：只有 attempt_i 匹配第 i 个输出才得分，错位不得分。
    # 返回 (总分, 正确数, 应预测总数, 每输出分值, 逐任务明细)。
    if submission is None:
        submission = load_json(SUBMISSION_FILE)
    if solutions is None:
        solutions = load_json(SOLUTIONS_FILE)

    total_required = sum(len(v) for v in solutions.values())
    if total_required == 0:
        return 0.0, 0, 0, 0.0, {}
    per_output = 100.0 / total_required

    correct = 0
    task_detail = {}
    for tid, truths in solutions.items():
        n = len(truths)
        pred = submission.get(tid)
        task_correct = 0
        for i in range(n):
            truth = _normalize_grid(truths[i])
            if n == 1:
                a1 = _normalize_grid(_get_attempt(pred, 0))
                a2 = _normalize_grid(_get_attempt(pred, 1))
                hit = (a1 is not None and a1 == truth) or (a2 is not None and a2 == truth)
            else:
                a = _normalize_grid(_get_attempt(pred, i))
                hit = a is not None and a == truth
            if hit:
                correct += 1
                task_correct += 1
        task_detail[tid] = task_correct
    score = per_output * correct
    return score, correct, total_required, per_output, task_detail


def score_existing(submission_file=None, score_file=None):
    # Score the current submission.json, print the report, and write score.json.
    # 给当前的 submission.json 打分，把报告打印出来，并写入 score.json。
    sub = load_json(submission_file or SUBMISSION_FILE)
    if score_file is None:
        score_file = SCORE_FILE
    score, correct, total, per, detail = score_submission(submission=sub)

    report = {
        "mode": "eval",
        "score": round(score, 4),
        "correct_outputs": correct,
        "total_outputs": total,
        "per_output_score": round(per, 6),
        "task_detail": detail,
    }
    save_json(report, score_file)

    print("=" * 60)
    print(f"[score] 100 points split across {total} outputs, {per:.4f} points each")
    print(f"[score] outputs predicted correctly: {correct}/{total}")
    print(f"[score] total: {score:.4f} / 100")
    print(f"[score] written to: {score_file}")
    print("=" * 60)
    return score, correct, total


# ===========================================================================
# 10) Entry points
# 10) 入口
# ===========================================================================

def run(mode=None, n=None):
    # Pre-fill -> solve -> rebuild + repair -> (eval mode) score.
    # 预填充 -> 求解 -> 重建 + 修复 ->（评估模式）打分。
    if mode is None:
        mode = MODE
    if n is None:
        n = N_PARALLEL
    if mode not in ("eval", "submit"):
        raise ValueError("mode must be 'eval' or 'submit'")

    t_start = time.perf_counter()
    challenges = load_json(CHALLENGES_FILE)
    print(f"tasks: {len(challenges)}  (from {CHALLENGES_FILE})")
    print(f"deadline: started {time.strftime('%H:%M:%S', time.localtime(RUN_START_TS))}, "
          f"{DEADLINE_MIN} min -> "
          f"{time.strftime('%H:%M:%S', time.localtime(deadline_ts()))}, "
          f"{_fmt_left()} left")

    # 1) A complete, valid submission exists before inference starts.
    # 1) 推理开始之前，先有一份完整合法的提交。
    submission = aggregate_to_submission(challenges=challenges)
    print(f"[run] pre-filled submission written before inference: {SUBMISSION_FILE}")

    # 2) Solve; each success overwrites its own slots in place.
    # 2) 求解；每成功一个就就地覆盖掉它自己的槽位。
    solve_challenges(challenges=challenges, n_parallel=n, submission=submission)

    # 3) Rebuild from the result files and run the final repair pass — reached whether
    #    the run finished normally or was cut off by the deadline.
    # 3) 从结果文件重建，并跑最终的兜底修复 —— 无论正常跑完还是被截止时间掐断都会走到。
    aggregate_to_submission(challenges=challenges)
    print("[run] submission rebuilt and verified after solving")

    if mode == "eval":
        if not os.path.exists(SOLUTIONS_FILE):
            print(f"[run] solutions file {SOLUTIONS_FILE} not found; cannot score "
                  f"(use submit mode, or supply the file)")
            return
        score_existing()
    print(f"[run] total wall clock: {_fmt_sec(time.perf_counter() - t_start)}")


def main(argv=None):
    import argparse
    p = argparse.ArgumentParser(description="ARC-AGI LLM solve/aggregate/score pipeline")
    p.add_argument("--mode", choices=["eval", "submit"], default=None,
                   help="eval: solve + aggregate + score; submit: solve + aggregate only")
    p.add_argument("--n", type=int, default=None, help="parallel tasks (default: N_PARALLEL, 16)")
    p.add_argument("--only-aggregate", action="store_true",
                   help="re-aggregate submission.json from existing output/*.json; no chat calls")
    p.add_argument("--only-score", action="store_true",
                   help="score the existing submission.json; no chat calls")
    args, unknown = p.parse_known_args(argv)
    if unknown:
        print(f"[warn] ignoring unrecognized arguments: {unknown}")

    if args.only_aggregate:
        aggregate_to_submission()
    elif args.only_score:
        score_existing()
    else:
        run(mode=args.mode, n=args.n)


if __name__ == "__main__":
    # Inside Jupyter/ipykernel, sys.argv holds the kernel's own arguments
    # (e.g. -f kernel-*.json), so argparse must not run automatically. Only run it
    # for a real terminal invocation: python arc_pipeline.py ...
    # 在 Jupyter/ipykernel 里，sys.argv 装的是内核自己的参数（如 -f kernel-*.json），
    # 所以不能自动跑 argparse。只在真正的终端调用时才执行：
    # python arc_pipeline.py ...
    try:
        get_ipython()
        _IN_IPYTHON = True
    except NameError:
        _IN_IPYTHON = False
    if not _IN_IPYTHON:
        main()


---


# 5. Run it

The whole pipeline is one call:

```python
run(mode='eval', n=32)
```

`mode` picks between scoring (`'eval'`) and producing a submission only (`'submit'`),
and `n` sets how many tasks are solved concurrently.

This is also where the deadline clock is reported, so you can see how much of the
budget is left before committing the run. Everything else — modes, tuning constants,
output files, resuming, the fallback, and troubleshooting — is documented in
**How to use** at the top of this notebook.

# 5. 运行

整条流水线就是一次调用：

```python
run(mode='eval', n=32)
```

`mode` 在"打分"（`'eval'`）和"只产出提交文件"（`'submit'`）之间选择，
`n` 设定同时求解多少个任务。

这里也会打印时限状态，所以在真正开跑之前你能看到还剩多少预算。其余内容 ——
模式、调参常量、输出文件、断点续跑、兜底机制、故障排查 —— 都写在本 notebook
开头的 **使用方法** 里。


In [ ]:
# Pick a mode and a concurrency level. See "How to use" at the top of the notebook.
# 选择模式和并发数。详见 notebook 开头的"How to use"。
# run(mode='eval', n=16)
run(mode='submit', n=16)
